# ================================================
# AOA++-Optimized Balanced Random Forest (BRF)
# for Leak-Safe Heart Disease Prediction
# Final reproducible notebook for Heliyon submission
# ================================================


# ==============================================================================
# PART 1: SETUP, DATA LOADING, PREPROCESSING & VISUALIZATION
# ==============================================================================

In [ ]:
# ===== Block 1 — Imports & Global Settings (Fully Anchored for Reproducibility across Servers) =====

import numpy as np
import pandas as pd
import os
import random
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from scipy import sparse
import warnings

from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline as SkPipeline
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer

from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score, train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    confusion_matrix,
)

from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.combine import SMOTEENN

from xgboost import XGBClassifier
from skopt import BayesSearchCV                                      

# ------------------------------------------------------------------------------
# # ===== Block 1 — Imports & Global Settings (Fully Anchored for Reproducibility across Servers) =====

import numpy as np
import pandas as pd
import os
import random
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from scipy import sparse
import warnings

from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline as SkPipeline
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer

from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score, train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    confusion_matrix,
)

from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.combine import SMOTEENN

from xgboost import XGBClassifier
from skopt import BayesSearchCV                                      

# ------------------------------------------------------------------------------
# # ===== Block 1 — Imports & Global Settings (Fully Anchored for Reproducibility across Servers) =====

import numpy as np
import pandas as pd
import os
import random
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from scipy import sparse
import warnings

from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline as SkPipeline
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer

from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score, train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    confusion_matrix,
)

from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.combine import SMOTEENN

from xgboost import XGBClassifier
from skopt import BayesSearchCV                                      


# Reproducibility settings: fixed seeds and limited CPU threading

os.environ['PYTHONHASHSEED'] = str(42)
random.seed(42)
np.random.seed(42)



os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'


sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300


warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*IProgress.*")

print("🎯 Global Random State and Hardware Multi-threading locked to 42!")
print("✅ Block 1 successfully adapted. All core libraries loaded and secured.")
# ------------------------------------------------------------------------------

os.environ['PYTHONHASHSEED'] = str(42)
random.seed(42)
np.random.seed(42)



os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'


sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300


warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*IProgress.*")

print("🎯 Global Random State and Hardware Multi-threading locked to 42!")
print("✅ Block 1 successfully adapted. All core libraries loaded and secured.")
# ------------------------------------------------------------------------------

os.environ['PYTHONHASHSEED'] = str(42)
random.seed(42)
np.random.seed(42)



os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'


sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300


warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*IProgress.*")

print("🎯 Global Random State and Hardware Multi-threading locked to 42!")
print("✅ Block 1 successfully adapted. All core libraries loaded and secured.")

In [ ]:
# ===== Block 2: Load Data & Define Columns =====
url = "https://raw.githubusercontent.com/Harikishan63/Data-Learnings-/main/heart_disease_uci.csv"
df = pd.read_csv(url)

target_candidates = [c for c in df.columns if c.lower() in ["target","output","disease","num"]]
if not target_candidates:
    raise ValueError("Target column not found")
target_col = target_candidates[0]

y = df[target_col].copy()
if y.nunique() > 2:
    y = (y > 0).astype(int)

drop_cols = [c for c in ["id"] if c in df.columns]
X = df.drop(columns=[target_col] + drop_cols).copy()

likely_categorical = set(['sex','cp','fbs','restecg','exang','slope','ca','thal','dataset'])
cat_cols = [c for c in X.columns if (c.lower() in likely_categorical) or (X[c].dtype == 'object')]
num_cols = [c for c in X.columns if c not in cat_cols]

print("✅ Data shape:", X.shape)
print("✅ Target distribution:", y.value_counts(normalize=True).round(3).to_dict())
print("Numeric cols:", num_cols)
print("Categorical cols:", cat_cols)


In [ ]:
# ===== Block 3: Exploratory Data Analysis (EDA) =====
from sklearn.feature_selection import mutual_info_classif
from scipy import stats

target_col = [c for c in df.columns if c.lower() in ["target","output","disease","num"]][0]
y = df[target_col].copy()

if y.nunique() > 2:
    y = (y > 0).astype(int)

X = df.drop(columns=[target_col]).copy()

# Define categorical and numeric columns
likely_categorical = set([
    'sex', 'cp', 'fbs', 'restecg',
    'exang', 'slope', 'ca', 'thal',
    'dataset'
])

cat_cols = [
    c for c in X.columns
    if (c.lower() in likely_categorical)
    or (X[c].dtype == 'object')
]

num_cols = [
    c for c in X.columns
    if c not in cat_cols and c.lower() != "id"
]
# =========================

print("✅ Data loaded.")
print("X shape:", X.shape)
print("y distribution:", y.value_counts().to_dict())
print("Numeric cols:", num_cols)
print("Categorical cols:", cat_cols)

plt.figure(figsize=(5,4))
sns.countplot(x=y, palette="Set2")
plt.title("Target Distribution (0=Healthy, 1=Disease)")
plt.show()

X[num_cols].hist(
    bins=20,
    figsize=(12,8),
    color="skyblue",
    edgecolor="black"
)
plt.suptitle("Histograms of Numeric Features")
plt.show()

plt.figure(figsize=(12,6))
sns.boxplot(data=X[num_cols], palette="Set3")
plt.title("Boxplots of Numeric Features")
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(8,6))
sns.heatmap(
    X[num_cols].corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Correlation Heatmap (Numeric Features)")
plt.show()

sns.pairplot(
    pd.concat([X[num_cols], y.rename("target")], axis=1),
    hue="target",
    diag_kind="kde",
    corner=True
)
plt.suptitle("Pairplot of Selected Numeric Features", y=1.02)
plt.show()

for c in cat_cols:
    plt.figure(figsize=(6,4))
    sns.countplot(
        x=X[c],
        hue=y,
        palette="Set1"
    )
    plt.title(f"{c} vs Target")
    plt.show()

imputer = SimpleImputer(strategy="median")
X_num_imp = imputer.fit_transform(X[num_cols])

mi = mutual_info_classif(
    X_num_imp,
    y,
    random_state=42
)

mi_series = pd.Series(
    mi,
    index=num_cols
).sort_values(ascending=False)

plt.figure(figsize=(8,6))
sns.barplot(
    x=mi_series.values,
    y=mi_series.index,
    color="salmon"
)
plt.title("Top Features by Mutual Information")
plt.xlabel("MI Score")
plt.show()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

lr = LogisticRegression(
    max_iter=200,
    class_weight="balanced"
)

oof = cross_val_predict(
    lr,
    X_num_imp,
    y,
    cv=cv,
    method="predict_proba"
)[:,1]

print(
    "Baseline Logistic Regression AUC:",
    roc_auc_score(y, oof)
)

for c in num_cols:
    stat, p = stats.shapiro(
        X[c].dropna()
    )
    print(
        f"Shapiro test for {c}: "
        f"p={p:.4f} -> "
        f"{'Not Normal' if p < 0.05 else 'Normal'}"
    )

In [ ]:
# ===== Block 4: Pipeline & Utils =====

from sklearn.pipeline import Pipeline as SkPipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

def build_pipeline(model, num_cols, cat_cols, sampler=None):
    num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("scaler", RobustScaler())])
    cat_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))])
    pre = ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)])
    steps = [("pre", pre)]
    if sampler is not None:
        steps.append(("sampler", sampler))
    steps.append(("model", model))
    return ImbPipeline(steps)

def evaluate_pipeline(pipe, X, y, cv):
    proba = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:,1]
    preds_05 = (proba >= 0.5).astype(int)
    metrics = {
        "accuracy": accuracy_score(y, preds_05),
        "roc_auc": roc_auc_score(y, proba),
        "f1": f1_score(y, preds_05),
        "precision": precision_score(y, preds_05),
        "recall": recall_score(y, preds_05)
    }
    RocCurveDisplay.from_predictions(y, proba); plt.title("ROC (OOF)"); plt.show()
    PrecisionRecallDisplay.from_predictions(y, proba); plt.title("PR (OOF)"); plt.show()
    return proba, metrics

def find_best_threshold(y_true, proba, metric="f1"):
    thresholds = np.linspace(0.05, 0.95, 37)
    best_t, best_val = 0.5, -np.inf
    for t in thresholds:
        preds = (proba >= t).astype(int)
        if metric == "f1":
            val = f1_score(y_true, preds)
        elif metric == "balanced_acc":
            from sklearn.metrics import balanced_accuracy_score
            val = balanced_accuracy_score(y_true, preds)
        else:
            val = f1_score(y_true, preds)
        if val > best_val:
            best_val, best_t = val, t
    return best_t, best_val


# ==============================================================================
# PART 2: HYPERPARAMETER OPTIMIZATION (AOA & AOA++)
# ==============================================================================

In [ ]:
# # ===== Block 5: AOA Optimization + Final Modeling + Evaluation (max_iter=5 , pop_size=8) =====

# def aoa_optimize_rf_simple(X, y, num_cols, cat_cols, max_iter=5, pop_size=8, seed=42):
#     rng = np.random.RandomState(seed)
#     def sample_candidate():
#         return {
#             "n_estimators": int(rng.randint(150, 601)),
#             "max_depth": rng.choice([None, 3, 5, 7, 10]),
#             "min_samples_split": int(rng.randint(2, 13)),
#             "min_samples_leaf": int(rng.randint(1, 7))
#         }
#     def build_rf(p):
#         return RandomForestClassifier(
#             n_estimators=p["n_estimators"],
#             max_depth=p["max_depth"],
#             min_samples_split=p["min_samples_split"],
#             min_samples_leaf=p["min_samples_leaf"],
#             n_jobs=-1,
#             random_state=seed
#         )
#     def build_brf(p):
#         return BalancedRandomForestClassifier(
#             n_estimators=p["n_estimators"],
#             max_depth=p["max_depth"],
#             min_samples_split=p["min_samples_split"],
#             min_samples_leaf=p["min_samples_leaf"],
#             random_state=seed,
#             n_jobs=-1
#         )
#     def fitness(p, use_brf=False, use_sampler=False):
#         if use_brf:
#             estimator = build_brf(p)
#             sampler = SMOTEENN(random_state=seed) if use_sampler else None
#             pipe = build_pipeline(estimator, num_cols, cat_cols, sampler=sampler)
#         else:
#             estimator = build_rf(p)
#             pipe = build_pipeline(estimator, num_cols, cat_cols, sampler=SMOTEENN(random_state=seed))
#         cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
#         oof_proba = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
#         oof_pred = (oof_proba >= 0.5).astype(int)
#         auc = roc_auc_score(y, oof_proba)
#         f1 = f1_score(y, oof_pred)
#         recall = recall_score(y, oof_pred)
#         acc = accuracy_score(y, oof_pred)
#         precision = precision_score(y, oof_pred)
#         score = 0.4 * auc + 0.4 * recall + 0.1 * f1 + 0.1 * acc
#         return score, {"auc": auc, "acc": acc, "f1": f1, "recall": recall, "precision": precision}
    
#     # Optimize for RF with SMOTEENN
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_rf, best_score_rf, best_metrics_rf = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=False)
#             if sc > best_score_rf:
#                 best_rf, best_score_rf, best_metrics_rf = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (RF+SMOTEENN) | Score={best_score_rf:.4f} | AUC={best_metrics_rf['auc']:.4f} | F1={best_metrics_rf['f1']:.4f} | ACC={best_metrics_rf['acc']:.4f} | Recall={best_metrics_rf['recall']:.4f}")

#     # Optimize for BRF without SMOTEENN (separate loop)
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_brf, best_score_brf, best_metrics_brf = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=True, use_sampler=False)
#             if sc > best_score_brf:
#                 best_brf, best_score_brf, best_metrics_brf = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (BRF) | Score={best_score_brf:.4f} | AUC={best_metrics_brf['auc']:.4f} | F1={best_metrics_brf['f1']:.4f} | ACC={best_metrics_brf['acc']:.4f} | Recall={best_metrics_brf['recall']:.4f}")
    
#     # Optimize for BRF with SMOTEENN (separate loop)
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_brf_smoteen, best_score_brf_smoteen, best_metrics_brf_smoteen = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=True, use_sampler=True)
#             if sc > best_score_brf_smoteen:
#                 best_brf_smoteen, best_score_brf_smoteen, best_metrics_brf_smoteen = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (BRF+SMOTEENN) | Score={best_score_brf_smoteen:.4f} | AUC={best_metrics_brf_smoteen['auc']:.4f} | F1={best_metrics_brf_smoteen['f1']:.4f} | ACC={best_metrics_brf_smoteen['acc']:.4f} | Recall={best_metrics_brf_smoteen['recall']:.4f}")

#     return (best_rf, best_metrics_rf), (best_brf, best_metrics_brf), (best_brf_smoteen, best_metrics_brf_smoteen)

# # اجرای بهینه‌سازی پارامترها
# (best_params_rf, best_mets_rf), (best_params_brf, best_mets_brf), (best_params_brf_smoteen, best_mets_brf_smoteen) = aoa_optimize_rf_simple(X, y, num_cols, cat_cols, max_iter=5, pop_size=8)

# print("\nBest RF params (AOA Simple with SMOTEENN):", best_params_rf)
# print("Best (AOA Simple with SMOTEENN) metrics (CV 10x1):", best_mets_rf)

# print("\nBest BRF params (AOA Simple without SMOTEENN):", best_params_brf)
# print("Best (AOA Simple without SMOTEENN) metrics (CV 10x1):", best_mets_brf)

# print("\nBest BRF params (AOA Simple with SMOTEENN):", best_params_brf_smoteen)
# print("Best (AOA Simple with SMOTEENN) metrics (CV 10x1):", best_mets_brf_smoteen)

# # ساخت مدل‌ها با پارامتر بهتر و pipeline
# rf_final = RandomForestClassifier(**best_params_rf, random_state=42, n_jobs=-1)
# pipe_final = build_pipeline(rf_final, num_cols, cat_cols, sampler=SMOTEENN(random_state=42))

# brf_final = BalancedRandomForestClassifier(**best_params_brf, random_state=42, n_jobs=-1)
# pipe_brf = build_pipeline(brf_final, num_cols, cat_cols, sampler=None)

# brf_final_smoteen = BalancedRandomForestClassifier(**best_params_brf_smoteen, random_state=42, n_jobs=-1)
# pipe_brf_smoteen = build_pipeline(brf_final_smoteen, num_cols, cat_cols, sampler=SMOTEENN(random_state=42))

# # تقسیم داده
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# # آموزش مدل‌ها
# pipe_final.fit(X_train, y_train)
# pipe_brf.fit(X_train, y_train)
# pipe_brf_smoteen.fit(X_train, y_train)

# # تابع محاسبه متریک روی train (OOF)
# def calc_metrics(pipe, X, y):
#     proba = cross_val_predict(pipe, X, y, cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
#                              method="predict_proba", n_jobs=-1)[:, 1]
#     preds = (proba >= 0.5).astype(int)
#     return {
#         "accuracy": accuracy_score(y, preds),
#         "roc_auc": roc_auc_score(y, proba),
#         "recall": recall_score(y, preds),
#         "precision": precision_score(y, preds),
#         "f1": f1_score(y, preds)
#     }

# # محاسبه متریک روی داده آموزش (train)
# metrics_train_rf = calc_metrics(pipe_final, X_train, y_train)
# metrics_train_brf = calc_metrics(pipe_brf, X_train, y_train)
# metrics_train_brf_smoteen = calc_metrics(pipe_brf_smoteen, X_train, y_train)

# # تابع محاسبه متریک و خروجی احتمال روی test
# def calc_metrics_test(pipe, X, y):
#     proba = pipe.predict_proba(X)[:, 1]
#     preds = (proba >= 0.5).astype(int)
#     return {
#         "accuracy": accuracy_score(y, preds),
#         "roc_auc": roc_auc_score(y, proba),
#         "recall": recall_score(y, preds),
#         "precision": precision_score(y, preds),
#         "f1": f1_score(y, preds)
#     }, proba

# # محاسبه متریک و احتمال روی داده تست (test)
# metrics_test_rf, proba_test_rf = calc_metrics_test(pipe_final, X_test, y_test)
# metrics_test_brf, proba_test_brf = calc_metrics_test(pipe_brf, X_test, y_test)
# metrics_test_brf_smoteen, proba_test_brf_smoteen = calc_metrics_test(pipe_brf_smoteen, X_test, y_test)

# # چاپ متریک‌ها (دسته‌بندی)
# print("\nFinal RF+SMOTEENN (Train/Oof):")
# for k, v in metrics_train_rf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal RF+SMOTEENN (Test):")
# for k, v in metrics_test_rf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF (Train/Oof):")
# for k, v in metrics_train_brf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF (Test):")
# for k, v in metrics_test_brf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF+SMOTEENN (Train/Oof):")
# for k, v in metrics_train_brf_smoteen.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF+SMOTEENN (Test):")
# for k, v in metrics_test_brf_smoteen.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# # رسم نمودار ROC و Precision-Recall فقط روی داده تست
# plt.figure(figsize=(14, 6))

# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_rf).plot()
# plt.title("ROC Curve RF (Test)")

# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_rf).plot()
# plt.title("Precision-Recall Curve RF (Test)")

# plt.show()

# plt.figure(figsize=(14, 6))

# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_brf).plot()
# plt.title("ROC Curve BRF (Test)")

# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_brf).plot()
# plt.title("Precision-Recall Curve BRF (Test)")

# plt.show()

# plt.figure(figsize=(14, 6))

# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_brf_smoteen).plot()
# plt.title("ROC Curve BRF+SMOTEENN (Test)")

# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_brf_smoteen).plot()
# plt.title("Precision-Recall Curve BRF+SMOTEENN (Test)")

# plt.show()


In [ ]:
# # ===== Block 6: AOA++ Optimization + Final Modeling + Evaluation (max_iter=5, pop_size=8) =====


# def aoa_optimize_rf_plusplus(X, y, num_cols, cat_cols, max_iter=5, pop_size=8, seed=42):
#     rng = np.random.RandomState(seed)
#     def sample_candidate():
#         return {
#             "n_estimators": int(rng.randint(150, 601)),
#             "max_depth": rng.choice([None, 3, 5, 7, 10]),
#             "min_samples_split": int(rng.randint(2, 13)),
#             "min_samples_leaf": int(rng.randint(1, 7)),
#             "max_features": rng.choice(['sqrt', 'log2', None]),
#             "bootstrap": bool(rng.randint(0,2)),
#             "class_weight": rng.choice([None, 'balanced'])
#         }
#     def build_rf(p):
#         return RandomForestClassifier(
#             n_estimators=p["n_estimators"],
#             max_depth=p["max_depth"],
#             min_samples_split=p["min_samples_split"],
#             min_samples_leaf=p["min_samples_leaf"],
#             max_features=p["max_features"],
#             bootstrap=p["bootstrap"],
#             class_weight=p["class_weight"],
#             n_jobs=-1,
#             random_state=seed
#         )
#     def build_brf(p):
#         # class_weight و bootstrap در BRF پشتیبانی نمی‌شود
#         return BalancedRandomForestClassifier(
#             n_estimators=p["n_estimators"],
#             max_depth=p["max_depth"],
#             min_samples_split=p["min_samples_split"],
#             min_samples_leaf=p["min_samples_leaf"],
#             max_features=p["max_features"],
#             random_state=seed,
#             n_jobs=-1
#         )
#     def fitness(p, use_brf=False, use_sampler=False):
#         if use_brf:
#             estimator = build_brf(p)
#             sampler = SMOTEENN(random_state=seed) if use_sampler else None
#             pipe = build_pipeline(estimator, num_cols, cat_cols, sampler=sampler)
#         else:
#             estimator = build_rf(p)
#             pipe = build_pipeline(estimator, num_cols, cat_cols, sampler=SMOTEENN(random_state=seed))
#         cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
#         oof_proba = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
#         oof_pred = (oof_proba >= 0.5).astype(int)
#         auc = roc_auc_score(y, oof_proba)
#         f1 = f1_score(y, oof_pred)
#         recall = recall_score(y, oof_pred)
#         acc = accuracy_score(y, oof_pred)
#         precision = precision_score(y, oof_pred)
#         score = 0.4 * auc + 0.4 * recall + 0.1 * f1 + 0.1 * acc
#         return score, {"auc": auc, "acc": acc, "f1": f1, "recall": recall, "precision": precision}

#     # RF+SMOTEENN
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_rf, best_score_rf, best_metrics_rf = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             if rng.rand() < 0.4:
#                 c["max_features"] = rng.choice(["sqrt", "log2", None])
#             if rng.rand() < 0.3:
#                 c["bootstrap"] = not c["bootstrap"]
#             if rng.rand() < 0.3:
#                 c["class_weight"] = rng.choice([None, "balanced"])
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=False)
#             if sc > best_score_rf:
#                 best_rf, best_score_rf, best_metrics_rf = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (RF+SMOTEENN) | Score={best_score_rf:.4f} | AUC={best_metrics_rf['auc']:.4f} | F1={best_metrics_rf['f1']:.4f} | ACC={best_metrics_rf['acc']:.4f} | Recall={best_metrics_rf['recall']:.4f}")

#     # BRF بدون SMOTEENN
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_brf, best_score_brf, best_metrics_brf = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             if rng.rand() < 0.4:
#                 c["max_features"] = rng.choice(["sqrt", "log2", None])
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=True, use_sampler=False)
#             if sc > best_score_brf:
#                 best_brf, best_score_brf, best_metrics_brf = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (BRF) | Score={best_score_brf:.4f} | AUC={best_metrics_brf['auc']:.4f} | F1={best_metrics_brf['f1']:.4f} | ACC={best_metrics_brf['acc']:.4f} | Recall={best_metrics_brf['recall']:.4f}")

#     # BRF+SMOTEENN
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_brf_smoteen, best_score_brf_smoteen, best_metrics_brf_smoteen = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             if rng.rand() < 0.4:
#                 c["max_features"] = rng.choice(["sqrt", "log2", None])
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=True, use_sampler=True)
#             if sc > best_score_brf_smoteen:
#                 best_brf_smoteen, best_score_brf_smoteen, best_metrics_brf_smoteen = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (BRF+SMOTEENN) | Score={best_score_brf_smoteen:.4f} | AUC={best_metrics_brf_smoteen['auc']:.4f} | F1={best_metrics_brf_smoteen['f1']:.4f} | ACC={best_metrics_brf_smoteen['acc']:.4f} | Recall={best_metrics_brf_smoteen['recall']:.4f}")

#     return (best_rf, best_metrics_rf), (best_brf, best_metrics_brf), (best_brf_smoteen, best_metrics_brf_smoteen)

# # اجرای بهینه‌سازی پارامترها به روش پیشرفته‌تر (AOA++)
# (best_params_rf, best_mets_rf), (best_params_brf, best_mets_brf), (best_params_brf_smoteen, best_mets_brf_smoteen) = aoa_optimize_rf_plusplus(X, y, num_cols, cat_cols, max_iter=5, pop_size=8)

# print("\nBest RF params (AOA++):", best_params_rf)
# print("Best (AOA++) metrics (CV 10x1):", best_mets_rf)

# print("\nBest BRF params (AOA++):", best_params_brf)
# print("Best (AOA++) metrics (CV 10x1):", best_mets_brf)

# print("\nBest BRF+SMOTEENN params (AOA++):", best_params_brf_smoteen)
# print("Best (AOA++ with SMOTEENN) metrics (CV 10x1):", best_mets_brf_smoteen)

# # ساخت مدل‌ها با پارامتر بهتر و pipeline
# rf_final = RandomForestClassifier(**best_params_rf, random_state=42, n_jobs=-1)
# pipe_final = build_pipeline(rf_final, num_cols, cat_cols, sampler=SMOTEENN(random_state=42))

# brf_final = BalancedRandomForestClassifier(**{k: v for k, v in best_params_brf.items() if k not in ['class_weight', 'bootstrap']}, random_state=42, n_jobs=-1)
# pipe_brf = build_pipeline(brf_final, num_cols, cat_cols, sampler=None)

# brf_final_smoteen = BalancedRandomForestClassifier(**{k: v for k, v in best_params_brf_smoteen.items() if k not in ['class_weight', 'bootstrap']}, random_state=42, n_jobs=-1)
# pipe_brf_smoteen = build_pipeline(brf_final_smoteen, num_cols, cat_cols, sampler=SMOTEENN(random_state=42))

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
# pipe_final.fit(X_train, y_train)
# pipe_brf.fit(X_train, y_train)
# pipe_brf_smoteen.fit(X_train, y_train)

# # تابع و چاپ متریک‌ها مشابه نسخه ساده (کپی از کد قبلی)
# def calc_metrics(pipe, X, y):
#     proba = cross_val_predict(pipe, X, y, cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
#                              method="predict_proba", n_jobs=-1)[:, 1]
#     preds = (proba >= 0.5).astype(int)
#     return {
#         "accuracy": accuracy_score(y, preds),
#         "roc_auc": roc_auc_score(y, proba),
#         "recall": recall_score(y, preds),
#         "precision": precision_score(y, preds),
#         "f1": f1_score(y, preds)
#     }

# metrics_train_rf = calc_metrics(pipe_final, X_train, y_train)
# metrics_train_brf = calc_metrics(pipe_brf, X_train, y_train)
# metrics_train_brf_smoteen = calc_metrics(pipe_brf_smoteen, X_train, y_train)

# def calc_metrics_test(pipe, X, y):
#     proba = pipe.predict_proba(X)[:, 1]
#     preds = (proba >= 0.5).astype(int)
#     return {
#         "accuracy": accuracy_score(y, preds),
#         "roc_auc": roc_auc_score(y, proba),
#         "recall": recall_score(y, preds),
#         "precision": precision_score(y, preds),
#         "f1": f1_score(y, preds)
#     }, proba

# metrics_test_rf, proba_test_rf = calc_metrics_test(pipe_final, X_test, y_test)
# metrics_test_brf, proba_test_brf = calc_metrics_test(pipe_brf, X_test, y_test)
# metrics_test_brf_smoteen, proba_test_brf_smoteen = calc_metrics_test(pipe_brf_smoteen, X_test, y_test)

# # چاپ بلوکی و دسته‌بندی حرفه‌ای
# print("\nFinal RF+SMOTEENN (Train/Oof):")
# for k, v in metrics_train_rf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal RF+SMOTEENN (Test):")
# for k, v in metrics_test_rf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF (Train/Oof):")
# for k, v in metrics_train_brf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF (Test):")
# for k, v in metrics_test_brf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF+SMOTEENN (Train/Oof):")
# for k, v in metrics_train_brf_smoteen.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF+SMOTEENN (Test):")
# for k, v in metrics_test_brf_smoteen.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# # رسم نمودارهای ROC و PR برای test
# plt.figure(figsize=(14, 6))
# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_rf).plot()
# plt.title("ROC Curve RF (Test)")
# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_rf).plot()
# plt.title("Precision-Recall Curve RF (Test)")
# plt.show()

# plt.figure(figsize=(14, 6))
# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_brf).plot()
# plt.title("ROC Curve BRF (Test)")
# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_brf).plot()
# plt.title("Precision-Recall Curve BRF (Test)")
# plt.show()

# plt.figure(figsize=(14, 6))
# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_brf_smoteen).plot()
# plt.title("ROC Curve BRF+SMOTEENN (Test)")
# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_brf_smoteen).plot()
# plt.title("Precision-Recall Curve BRF+SMOTEENN (Test)")
# plt.show()


In [ ]:
# # ===== Block 7: AOA Optimization + Final Modeling + Evaluation (max_iter=10,popsize=16) =====

# def aoa_optimize_rf_simple(X, y, num_cols, cat_cols, max_iter=10, pop_size=16, seed=42):
#     rng = np.random.RandomState(seed)
#     def sample_candidate():
#         return {
#             "n_estimators": int(rng.randint(150, 601)),
#             "max_depth": rng.choice([None, 3, 5, 7, 10]),
#             "min_samples_split": int(rng.randint(2, 13)),
#             "min_samples_leaf": int(rng.randint(1, 7))
#         }
#     def build_rf(p):
#         return RandomForestClassifier(
#             n_estimators=p["n_estimators"],
#             max_depth=p["max_depth"],
#             min_samples_split=p["min_samples_split"],
#             min_samples_leaf=p["min_samples_leaf"],
#             n_jobs=-1,
#             random_state=seed
#         )
#     def build_brf(p):
#         return BalancedRandomForestClassifier(
#             n_estimators=p["n_estimators"],
#             max_depth=p["max_depth"],
#             min_samples_split=p["min_samples_split"],
#             min_samples_leaf=p["min_samples_leaf"],
#             random_state=seed,
#             n_jobs=-1
#         )
#     def fitness(p, use_brf=False, use_sampler=False):
#         if use_brf:
#             estimator = build_brf(p)
#             sampler = SMOTEENN(random_state=seed) if use_sampler else None
#             pipe = build_pipeline(estimator, num_cols, cat_cols, sampler=sampler)
#         else:
#             estimator = build_rf(p)
#             pipe = build_pipeline(estimator, num_cols, cat_cols, sampler=SMOTEENN(random_state=seed))
#         cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
#         oof_proba = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
#         oof_pred = (oof_proba >= 0.5).astype(int)
#         auc = roc_auc_score(y, oof_proba)
#         f1 = f1_score(y, oof_pred)
#         recall = recall_score(y, oof_pred)
#         acc = accuracy_score(y, oof_pred)
#         precision = precision_score(y, oof_pred)
#         score = 0.4 * auc + 0.4 * recall + 0.1 * f1 + 0.1 * acc
#         return score, {"auc": auc, "acc": acc, "f1": f1, "recall": recall, "precision": precision}
    
#     # Optimize for RF with SMOTEENN
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_rf, best_score_rf, best_metrics_rf = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=False)
#             if sc > best_score_rf:
#                 best_rf, best_score_rf, best_metrics_rf = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (RF+SMOTEENN) | Score={best_score_rf:.4f} | AUC={best_metrics_rf['auc']:.4f} | F1={best_metrics_rf['f1']:.4f} | ACC={best_metrics_rf['acc']:.4f} | Recall={best_metrics_rf['recall']:.4f}")

#     # Optimize for BRF without SMOTEENN (separate loop)
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_brf, best_score_brf, best_metrics_brf = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=True, use_sampler=False)
#             if sc > best_score_brf:
#                 best_brf, best_score_brf, best_metrics_brf = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (BRF) | Score={best_score_brf:.4f} | AUC={best_metrics_brf['auc']:.4f} | F1={best_metrics_brf['f1']:.4f} | ACC={best_metrics_brf['acc']:.4f} | Recall={best_metrics_brf['recall']:.4f}")
    
#     # Optimize for BRF with SMOTEENN (separate loop)
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_brf_smoteen, best_score_brf_smoteen, best_metrics_brf_smoteen = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=True, use_sampler=True)
#             if sc > best_score_brf_smoteen:
#                 best_brf_smoteen, best_score_brf_smoteen, best_metrics_brf_smoteen = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (BRF+SMOTEENN) | Score={best_score_brf_smoteen:.4f} | AUC={best_metrics_brf_smoteen['auc']:.4f} | F1={best_metrics_brf_smoteen['f1']:.4f} | ACC={best_metrics_brf_smoteen['acc']:.4f} | Recall={best_metrics_brf_smoteen['recall']:.4f}")

#     return (best_rf, best_metrics_rf), (best_brf, best_metrics_brf), (best_brf_smoteen, best_metrics_brf_smoteen)

# # اجرای بهینه‌سازی پارامترها
# (best_params_rf, best_mets_rf), (best_params_brf, best_mets_brf), (best_params_brf_smoteen, best_mets_brf_smoteen) = aoa_optimize_rf_simple(X, y, num_cols, cat_cols, max_iter=10, pop_size=16)

# print("\nBest RF params (AOA Simple with SMOTEENN):", best_params_rf)
# print("Best (AOA Simple with SMOTEENN) metrics (CV 10x1):", best_mets_rf)

# print("\nBest BRF params (AOA Simple without SMOTEENN):", best_params_brf)
# print("Best (AOA Simple without SMOTEENN) metrics (CV 10x1):", best_mets_brf)

# print("\nBest BRF params (AOA Simple with SMOTEENN):", best_params_brf_smoteen)
# print("Best (AOA Simple with SMOTEENN) metrics (CV 10x1):", best_mets_brf_smoteen)

# # ساخت مدل‌ها با پارامتر بهتر و pipeline
# rf_final = RandomForestClassifier(**best_params_rf, random_state=42, n_jobs=-1)
# pipe_final = build_pipeline(rf_final, num_cols, cat_cols, sampler=SMOTEENN(random_state=42))

# brf_final = BalancedRandomForestClassifier(**best_params_brf, random_state=42, n_jobs=-1)
# pipe_brf = build_pipeline(brf_final, num_cols, cat_cols, sampler=None)

# brf_final_smoteen = BalancedRandomForestClassifier(**best_params_brf_smoteen, random_state=42, n_jobs=-1)
# pipe_brf_smoteen = build_pipeline(brf_final_smoteen, num_cols, cat_cols, sampler=SMOTEENN(random_state=42))

# # تقسیم داده
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# # آموزش مدل‌ها
# pipe_final.fit(X_train, y_train)
# pipe_brf.fit(X_train, y_train)
# pipe_brf_smoteen.fit(X_train, y_train)

# # تابع محاسبه متریک روی train (OOF)
# def calc_metrics(pipe, X, y):
#     proba = cross_val_predict(pipe, X, y, cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
#                              method="predict_proba", n_jobs=-1)[:, 1]
#     preds = (proba >= 0.5).astype(int)
#     return {
#         "accuracy": accuracy_score(y, preds),
#         "roc_auc": roc_auc_score(y, proba),
#         "recall": recall_score(y, preds),
#         "precision": precision_score(y, preds),
#         "f1": f1_score(y, preds)
#     }

# # محاسبه متریک روی داده آموزش (train)
# metrics_train_rf = calc_metrics(pipe_final, X_train, y_train)
# metrics_train_brf = calc_metrics(pipe_brf, X_train, y_train)
# metrics_train_brf_smoteen = calc_metrics(pipe_brf_smoteen, X_train, y_train)

# # تابع محاسبه متریک و خروجی احتمال روی test
# def calc_metrics_test(pipe, X, y):
#     proba = pipe.predict_proba(X)[:, 1]
#     preds = (proba >= 0.5).astype(int)
#     return {
#         "accuracy": accuracy_score(y, preds),
#         "roc_auc": roc_auc_score(y, proba),
#         "recall": recall_score(y, preds),
#         "precision": precision_score(y, preds),
#         "f1": f1_score(y, preds)
#     }, proba

# # محاسبه متریک و احتمال روی داده تست (test)
# metrics_test_rf, proba_test_rf = calc_metrics_test(pipe_final, X_test, y_test)
# metrics_test_brf, proba_test_brf = calc_metrics_test(pipe_brf, X_test, y_test)
# metrics_test_brf_smoteen, proba_test_brf_smoteen = calc_metrics_test(pipe_brf_smoteen, X_test, y_test)

# # چاپ متریک‌ها (دسته‌بندی)
# print("\nFinal RF+SMOTEENN (Train/Oof):")
# for k, v in metrics_train_rf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal RF+SMOTEENN (Test):")
# for k, v in metrics_test_rf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF (Train/Oof):")
# for k, v in metrics_train_brf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF (Test):")
# for k, v in metrics_test_brf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF+SMOTEENN (Train/Oof):")
# for k, v in metrics_train_brf_smoteen.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF+SMOTEENN (Test):")
# for k, v in metrics_test_brf_smoteen.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# # رسم نمودار ROC و Precision-Recall فقط روی داده تست
# plt.figure(figsize=(14, 6))

# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_rf).plot()
# plt.title("ROC Curve RF (Test)")

# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_rf).plot()
# plt.title("Precision-Recall Curve RF (Test)")

# plt.show()

# plt.figure(figsize=(14, 6))

# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_brf).plot()
# plt.title("ROC Curve BRF (Test)")

# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_brf).plot()
# plt.title("Precision-Recall Curve BRF (Test)")

# plt.show()

# plt.figure(figsize=(14, 6))

# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_brf_smoteen).plot()
# plt.title("ROC Curve BRF+SMOTEENN (Test)")

# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_brf_smoteen).plot()
# plt.title("Precision-Recall Curve BRF+SMOTEENN (Test)")

# plt.show()


In [ ]:
# # ===== Block 8: AOA++ Optimization + Final Modeling + Evaluation (max_iter=10, pop_size=16) =====


# def aoa_optimize_rf_plusplus(X, y, num_cols, cat_cols, max_iter=10, pop_size=16, seed=42):
#     rng = np.random.RandomState(seed)
#     def sample_candidate():
#         return {
#             "n_estimators": int(rng.randint(150, 601)),
#             "max_depth": rng.choice([None, 3, 5, 7, 10]),
#             "min_samples_split": int(rng.randint(2, 13)),
#             "min_samples_leaf": int(rng.randint(1, 7)),
#             "max_features": rng.choice(['sqrt', 'log2', None]),
#             "bootstrap": bool(rng.randint(0,2)),
#             "class_weight": rng.choice([None, 'balanced'])
#         }
#     def build_rf(p):
#         return RandomForestClassifier(
#             n_estimators=p["n_estimators"],
#             max_depth=p["max_depth"],
#             min_samples_split=p["min_samples_split"],
#             min_samples_leaf=p["min_samples_leaf"],
#             max_features=p["max_features"],
#             bootstrap=p["bootstrap"],
#             class_weight=p["class_weight"],
#             n_jobs=-1,
#             random_state=seed
#         )
#     def build_brf(p):
#         # class_weight و bootstrap در BRF پشتیبانی نمی‌شود
#         return BalancedRandomForestClassifier(
#             n_estimators=p["n_estimators"],
#             max_depth=p["max_depth"],
#             min_samples_split=p["min_samples_split"],
#             min_samples_leaf=p["min_samples_leaf"],
#             max_features=p["max_features"],
#             random_state=seed,
#             n_jobs=-1
#         )
#     def fitness(p, use_brf=False, use_sampler=False):
#         if use_brf:
#             estimator = build_brf(p)
#             sampler = SMOTEENN(random_state=seed) if use_sampler else None
#             pipe = build_pipeline(estimator, num_cols, cat_cols, sampler=sampler)
#         else:
#             estimator = build_rf(p)
#             pipe = build_pipeline(estimator, num_cols, cat_cols, sampler=SMOTEENN(random_state=seed))
#         cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
#         oof_proba = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
#         oof_pred = (oof_proba >= 0.5).astype(int)
#         auc = roc_auc_score(y, oof_proba)
#         f1 = f1_score(y, oof_pred)
#         recall = recall_score(y, oof_pred)
#         acc = accuracy_score(y, oof_pred)
#         precision = precision_score(y, oof_pred)
#         score = 0.4 * auc + 0.4 * recall + 0.1 * f1 + 0.1 * acc
#         return score, {"auc": auc, "acc": acc, "f1": f1, "recall": recall, "precision": precision}

#     # RF+SMOTEENN
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_rf, best_score_rf, best_metrics_rf = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             if rng.rand() < 0.4:
#                 c["max_features"] = rng.choice(["sqrt", "log2", None])
#             if rng.rand() < 0.3:
#                 c["bootstrap"] = not c["bootstrap"]
#             if rng.rand() < 0.3:
#                 c["class_weight"] = rng.choice([None, "balanced"])
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=False)
#             if sc > best_score_rf:
#                 best_rf, best_score_rf, best_metrics_rf = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (RF+SMOTEENN) | Score={best_score_rf:.4f} | AUC={best_metrics_rf['auc']:.4f} | F1={best_metrics_rf['f1']:.4f} | ACC={best_metrics_rf['acc']:.4f} | Recall={best_metrics_rf['recall']:.4f}")

#     # BRF بدون SMOTEENN
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_brf, best_score_brf, best_metrics_brf = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             if rng.rand() < 0.4:
#                 c["max_features"] = rng.choice(["sqrt", "log2", None])
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=True, use_sampler=False)
#             if sc > best_score_brf:
#                 best_brf, best_score_brf, best_metrics_brf = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (BRF) | Score={best_score_brf:.4f} | AUC={best_metrics_brf['auc']:.4f} | F1={best_metrics_brf['f1']:.4f} | ACC={best_metrics_brf['acc']:.4f} | Recall={best_metrics_brf['recall']:.4f}")

#     # BRF+SMOTEENN
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_brf_smoteen, best_score_brf_smoteen, best_metrics_brf_smoteen = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             if rng.rand() < 0.4:
#                 c["max_features"] = rng.choice(["sqrt", "log2", None])
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=True, use_sampler=True)
#             if sc > best_score_brf_smoteen:
#                 best_brf_smoteen, best_score_brf_smoteen, best_metrics_brf_smoteen = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (BRF+SMOTEENN) | Score={best_score_brf_smoteen:.4f} | AUC={best_metrics_brf_smoteen['auc']:.4f} | F1={best_metrics_brf_smoteen['f1']:.4f} | ACC={best_metrics_brf_smoteen['acc']:.4f} | Recall={best_metrics_brf_smoteen['recall']:.4f}")

#     return (best_rf, best_metrics_rf), (best_brf, best_metrics_brf), (best_brf_smoteen, best_metrics_brf_smoteen)

# # اجرای بهینه‌سازی پارامترها به روش پیشرفته‌تر (AOA++)
# (best_params_rf, best_mets_rf), (best_params_brf, best_mets_brf), (best_params_brf_smoteen, best_mets_brf_smoteen) = aoa_optimize_rf_plusplus(X, y, num_cols, cat_cols, max_iter=10, pop_size=16)

# print("\nBest RF params (AOA++):", best_params_rf)
# print("Best (AOA++) metrics (CV 10x1):", best_mets_rf)

# print("\nBest BRF params (AOA++):", best_params_brf)
# print("Best (AOA++) metrics (CV 10x1):", best_mets_brf)

# print("\nBest BRF+SMOTEENN params (AOA++):", best_params_brf_smoteen)
# print("Best (AOA++ with SMOTEENN) metrics (CV 10x1):", best_mets_brf_smoteen)

# # ساخت مدل‌ها با پارامتر بهتر و pipeline
# rf_final = RandomForestClassifier(**best_params_rf, random_state=42, n_jobs=-1)
# pipe_final = build_pipeline(rf_final, num_cols, cat_cols, sampler=SMOTEENN(random_state=42))

# brf_final = BalancedRandomForestClassifier(**{k: v for k, v in best_params_brf.items() if k not in ['class_weight', 'bootstrap']}, random_state=42, n_jobs=-1)
# pipe_brf = build_pipeline(brf_final, num_cols, cat_cols, sampler=None)

# brf_final_smoteen = BalancedRandomForestClassifier(**{k: v for k, v in best_params_brf_smoteen.items() if k not in ['class_weight', 'bootstrap']}, random_state=42, n_jobs=-1)
# pipe_brf_smoteen = build_pipeline(brf_final_smoteen, num_cols, cat_cols, sampler=SMOTEENN(random_state=42))

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
# pipe_final.fit(X_train, y_train)
# pipe_brf.fit(X_train, y_train)
# pipe_brf_smoteen.fit(X_train, y_train)

# # تابع و چاپ متریک‌ها مشابه نسخه ساده (کپی از کد قبلی)
# def calc_metrics(pipe, X, y):
#     proba = cross_val_predict(pipe, X, y, cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
#                              method="predict_proba", n_jobs=-1)[:, 1]
#     preds = (proba >= 0.5).astype(int)
#     return {
#         "accuracy": accuracy_score(y, preds),
#         "roc_auc": roc_auc_score(y, proba),
#         "recall": recall_score(y, preds),
#         "precision": precision_score(y, preds),
#         "f1": f1_score(y, preds)
#     }

# metrics_train_rf = calc_metrics(pipe_final, X_train, y_train)
# metrics_train_brf = calc_metrics(pipe_brf, X_train, y_train)
# metrics_train_brf_smoteen = calc_metrics(pipe_brf_smoteen, X_train, y_train)

# def calc_metrics_test(pipe, X, y):
#     proba = pipe.predict_proba(X)[:, 1]
#     preds = (proba >= 0.5).astype(int)
#     return {
#         "accuracy": accuracy_score(y, preds),
#         "roc_auc": roc_auc_score(y, proba),
#         "recall": recall_score(y, preds),
#         "precision": precision_score(y, preds),
#         "f1": f1_score(y, preds)
#     }, proba

# metrics_test_rf, proba_test_rf = calc_metrics_test(pipe_final, X_test, y_test)
# metrics_test_brf, proba_test_brf = calc_metrics_test(pipe_brf, X_test, y_test)
# metrics_test_brf_smoteen, proba_test_brf_smoteen = calc_metrics_test(pipe_brf_smoteen, X_test, y_test)

# # چاپ بلوکی و دسته‌بندی حرفه‌ای
# print("\nFinal RF+SMOTEENN (Train/Oof):")
# for k, v in metrics_train_rf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal RF+SMOTEENN (Test):")
# for k, v in metrics_test_rf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF (Train/Oof):")
# for k, v in metrics_train_brf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF (Test):")
# for k, v in metrics_test_brf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF+SMOTEENN (Train/Oof):")
# for k, v in metrics_train_brf_smoteen.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF+SMOTEENN (Test):")
# for k, v in metrics_test_brf_smoteen.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# # رسم نمودارهای ROC و PR برای test
# plt.figure(figsize=(14, 6))
# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_rf).plot()
# plt.title("ROC Curve RF (Test)")
# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_rf).plot()
# plt.title("Precision-Recall Curve RF (Test)")
# plt.show()

# plt.figure(figsize=(14, 6))
# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_brf).plot()
# plt.title("ROC Curve BRF (Test)")
# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_brf).plot()
# plt.title("Precision-Recall Curve BRF (Test)")
# plt.show()

# plt.figure(figsize=(14, 6))
# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_brf_smoteen).plot()
# plt.title("ROC Curve BRF+SMOTEENN (Test)")
# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_brf_smoteen).plot()
# plt.title("Precision-Recall Curve BRF+SMOTEENN (Test)")
# plt.show()


In [ ]:
# # ===== Block 9: AOA Optimization + Final Modeling + Evaluation (max_iter=15, pop_size=24) =====

# def aoa_optimize_rf_simple(X, y, num_cols, cat_cols, max_iter=15, pop_size=24, seed=42):
#     rng = np.random.RandomState(seed)
#     def sample_candidate():
#         return {
#             "n_estimators": int(rng.randint(150, 601)),
#             "max_depth": rng.choice([None, 3, 5, 7, 10]),
#             "min_samples_split": int(rng.randint(2, 13)),
#             "min_samples_leaf": int(rng.randint(1, 7))
#         }
#     def build_rf(p):
#         return RandomForestClassifier(
#             n_estimators=p["n_estimators"],
#             max_depth=p["max_depth"],
#             min_samples_split=p["min_samples_split"],
#             min_samples_leaf=p["min_samples_leaf"],
#             n_jobs=-1,
#             random_state=seed
#         )
#     def build_brf(p):
#         return BalancedRandomForestClassifier(
#             n_estimators=p["n_estimators"],
#             max_depth=p["max_depth"],
#             min_samples_split=p["min_samples_split"],
#             min_samples_leaf=p["min_samples_leaf"],
#             random_state=seed,
#             n_jobs=-1
#         )
#     def fitness(p, use_brf=False, use_sampler=False):
#         if use_brf:
#             estimator = build_brf(p)
#             sampler = SMOTEENN(random_state=seed) if use_sampler else None
#             pipe = build_pipeline(estimator, num_cols, cat_cols, sampler=sampler)
#         else:
#             estimator = build_rf(p)
#             pipe = build_pipeline(estimator, num_cols, cat_cols, sampler=SMOTEENN(random_state=seed))
#         cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
#         oof_proba = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
#         oof_pred = (oof_proba >= 0.5).astype(int)
#         auc = roc_auc_score(y, oof_proba)
#         f1 = f1_score(y, oof_pred)
#         recall = recall_score(y, oof_pred)
#         acc = accuracy_score(y, oof_pred)
#         precision = precision_score(y, oof_pred)
#         score = 0.4 * auc + 0.4 * recall + 0.1 * f1 + 0.1 * acc
#         return score, {"auc": auc, "acc": acc, "f1": f1, "recall": recall, "precision": precision}
    
#     # Optimize for RF with SMOTEENN
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_rf, best_score_rf, best_metrics_rf = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=False)
#             if sc > best_score_rf:
#                 best_rf, best_score_rf, best_metrics_rf = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (RF+SMOTEENN) | Score={best_score_rf:.4f} | AUC={best_metrics_rf['auc']:.4f} | F1={best_metrics_rf['f1']:.4f} | ACC={best_metrics_rf['acc']:.4f} | Recall={best_metrics_rf['recall']:.4f}")

#     # Optimize for BRF without SMOTEENN (separate loop)
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_brf, best_score_brf, best_metrics_brf = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=True, use_sampler=False)
#             if sc > best_score_brf:
#                 best_brf, best_score_brf, best_metrics_brf = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (BRF) | Score={best_score_brf:.4f} | AUC={best_metrics_brf['auc']:.4f} | F1={best_metrics_brf['f1']:.4f} | ACC={best_metrics_brf['acc']:.4f} | Recall={best_metrics_brf['recall']:.4f}")
    
#     # Optimize for BRF with SMOTEENN (separate loop)
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_brf_smoteen, best_score_brf_smoteen, best_metrics_brf_smoteen = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=True, use_sampler=True)
#             if sc > best_score_brf_smoteen:
#                 best_brf_smoteen, best_score_brf_smoteen, best_metrics_brf_smoteen = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (BRF+SMOTEENN) | Score={best_score_brf_smoteen:.4f} | AUC={best_metrics_brf_smoteen['auc']:.4f} | F1={best_metrics_brf_smoteen['f1']:.4f} | ACC={best_metrics_brf_smoteen['acc']:.4f} | Recall={best_metrics_brf_smoteen['recall']:.4f}")

#     return (best_rf, best_metrics_rf), (best_brf, best_metrics_brf), (best_brf_smoteen, best_metrics_brf_smoteen)

# # اجرای بهینه‌سازی پارامترها
# (best_params_rf, best_mets_rf), (best_params_brf, best_mets_brf), (best_params_brf_smoteen, best_mets_brf_smoteen) = aoa_optimize_rf_simple(X, y, num_cols, cat_cols, max_iter=15, pop_size=24)

# print("\nBest RF params (AOA Simple with SMOTEENN):", best_params_rf)
# print("Best (AOA Simple with SMOTEENN) metrics (CV 10x1):", best_mets_rf)

# print("\nBest BRF params (AOA Simple without SMOTEENN):", best_params_brf)
# print("Best (AOA Simple without SMOTEENN) metrics (CV 10x1):", best_mets_brf)

# print("\nBest BRF params (AOA Simple with SMOTEENN):", best_params_brf_smoteen)
# print("Best (AOA Simple with SMOTEENN) metrics (CV 10x1):", best_mets_brf_smoteen)

# # ساخت مدل‌ها با پارامتر بهتر و pipeline
# rf_final = RandomForestClassifier(**best_params_rf, random_state=42, n_jobs=-1)
# pipe_final = build_pipeline(rf_final, num_cols, cat_cols, sampler=SMOTEENN(random_state=42))

# brf_final = BalancedRandomForestClassifier(**best_params_brf, random_state=42, n_jobs=-1)
# pipe_brf = build_pipeline(brf_final, num_cols, cat_cols, sampler=None)

# brf_final_smoteen = BalancedRandomForestClassifier(**best_params_brf_smoteen, random_state=42, n_jobs=-1)
# pipe_brf_smoteen = build_pipeline(brf_final_smoteen, num_cols, cat_cols, sampler=SMOTEENN(random_state=42))

# # تقسیم داده
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# # آموزش مدل‌ها
# pipe_final.fit(X_train, y_train)
# pipe_brf.fit(X_train, y_train)
# pipe_brf_smoteen.fit(X_train, y_train)

# # تابع محاسبه متریک روی train (OOF)
# def calc_metrics(pipe, X, y):
#     proba = cross_val_predict(pipe, X, y, cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
#                              method="predict_proba", n_jobs=-1)[:, 1]
#     preds = (proba >= 0.5).astype(int)
#     return {
#         "accuracy": accuracy_score(y, preds),
#         "roc_auc": roc_auc_score(y, proba),
#         "recall": recall_score(y, preds),
#         "precision": precision_score(y, preds),
#         "f1": f1_score(y, preds)
#     }

# # محاسبه متریک روی داده آموزش (train)
# metrics_train_rf = calc_metrics(pipe_final, X_train, y_train)
# metrics_train_brf = calc_metrics(pipe_brf, X_train, y_train)
# metrics_train_brf_smoteen = calc_metrics(pipe_brf_smoteen, X_train, y_train)

# # تابع محاسبه متریک و خروجی احتمال روی test
# def calc_metrics_test(pipe, X, y):
#     proba = pipe.predict_proba(X)[:, 1]
#     preds = (proba >= 0.5).astype(int)
#     return {
#         "accuracy": accuracy_score(y, preds),
#         "roc_auc": roc_auc_score(y, proba),
#         "recall": recall_score(y, preds),
#         "precision": precision_score(y, preds),
#         "f1": f1_score(y, preds)
#     }, proba

# # محاسبه متریک و احتمال روی داده تست (test)
# metrics_test_rf, proba_test_rf = calc_metrics_test(pipe_final, X_test, y_test)
# metrics_test_brf, proba_test_brf = calc_metrics_test(pipe_brf, X_test, y_test)
# metrics_test_brf_smoteen, proba_test_brf_smoteen = calc_metrics_test(pipe_brf_smoteen, X_test, y_test)

# # چاپ متریک‌ها (دسته‌بندی)
# print("\nFinal RF+SMOTEENN (Train/Oof):")
# for k, v in metrics_train_rf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal RF+SMOTEENN (Test):")
# for k, v in metrics_test_rf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF (Train/Oof):")
# for k, v in metrics_train_brf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF (Test):")
# for k, v in metrics_test_brf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF+SMOTEENN (Train/Oof):")
# for k, v in metrics_train_brf_smoteen.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF+SMOTEENN (Test):")
# for k, v in metrics_test_brf_smoteen.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# # رسم نمودار ROC و Precision-Recall فقط روی داده تست
# plt.figure(figsize=(14, 6))

# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_rf).plot()
# plt.title("ROC Curve RF (Test)")

# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_rf).plot()
# plt.title("Precision-Recall Curve RF (Test)")

# plt.show()

# plt.figure(figsize=(14, 6))

# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_brf).plot()
# plt.title("ROC Curve BRF (Test)")

# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_brf).plot()
# plt.title("Precision-Recall Curve BRF (Test)")

# plt.show()

# plt.figure(figsize=(14, 6))

# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_brf_smoteen).plot()
# plt.title("ROC Curve BRF+SMOTEENN (Test)")

# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_brf_smoteen).plot()
# plt.title("Precision-Recall Curve BRF+SMOTEENN (Test)")

# plt.show()


In [ ]:
# # ===== Block 10: AOA++ Optimization + Final Modeling + Evaluation (max_iter=15, pop_size=24) =====

# def aoa_optimize_rf_plusplus(X, y, num_cols, cat_cols, max_iter=15, pop_size=24, seed=42):
#     rng = np.random.RandomState(seed)
#     def sample_candidate():
#         return {
#             "n_estimators": int(rng.randint(150, 601)),
#             "max_depth": rng.choice([None, 3, 5, 7, 10]),
#             "min_samples_split": int(rng.randint(2, 13)),
#             "min_samples_leaf": int(rng.randint(1, 7)),
#             "max_features": rng.choice(['sqrt', 'log2', None]),
#             "bootstrap": bool(rng.randint(0,2)),
#             "class_weight": rng.choice([None, 'balanced'])
#         }
#     def build_rf(p):
#         return RandomForestClassifier(
#             n_estimators=p["n_estimators"],
#             max_depth=p["max_depth"],
#             min_samples_split=p["min_samples_split"],
#             min_samples_leaf=p["min_samples_leaf"],
#             max_features=p["max_features"],
#             bootstrap=p["bootstrap"],
#             class_weight=p["class_weight"],
#             n_jobs=-1,
#             random_state=seed
#         )
#     def build_brf(p):
#         return BalancedRandomForestClassifier(
#             n_estimators=p["n_estimators"],
#             max_depth=p["max_depth"],
#             min_samples_split=p["min_samples_split"],
#             min_samples_leaf=p["min_samples_leaf"],
#             max_features=p["max_features"],
#             random_state=seed,
#             n_jobs=-1
#         )
#     def fitness(p, use_brf=False, use_sampler=False):
#         if use_brf:
#             estimator = build_brf(p)
#             sampler = SMOTEENN(random_state=seed) if use_sampler else None
#             pipe = build_pipeline(estimator, num_cols, cat_cols, sampler=sampler)
#         else:
#             estimator = build_rf(p)
#             pipe = build_pipeline(estimator, num_cols, cat_cols, sampler=SMOTEENN(random_state=seed))
#         cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
#         oof_proba = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
#         oof_pred = (oof_proba >= 0.5).astype(int)
#         auc = roc_auc_score(y, oof_proba)
#         f1 = f1_score(y, oof_pred)
#         recall = recall_score(y, oof_pred)
#         acc = accuracy_score(y, oof_pred)
#         precision = precision_score(y, oof_pred)
#         score = 0.4 * auc + 0.4 * recall + 0.1 * f1 + 0.1 * acc
#         return score, {"auc": auc, "acc": acc, "f1": f1, "recall": recall, "precision": precision}
    
#     # Optimize for RF with SMOTEENN
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_rf, best_score_rf, best_metrics_rf = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             if rng.rand() < 0.4:
#                 c["max_features"] = rng.choice(["sqrt", "log2", None])
#             if rng.rand() < 0.3:
#                 c["bootstrap"] = not c["bootstrap"]
#             if rng.rand() < 0.3:
#                 c["class_weight"] = rng.choice([None, "balanced"])
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=False)
#             if sc > best_score_rf:
#                 best_rf, best_score_rf, best_metrics_rf = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (RF+SMOTEENN) | Score={best_score_rf:.4f} | AUC={best_metrics_rf['auc']:.4f} | F1={best_metrics_rf['f1']:.4f} | ACC={best_metrics_rf['acc']:.4f} | Recall={best_metrics_rf['recall']:.4f}")

#     # Optimize for BRF without SMOTEENN (separate loop)
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_brf, best_score_brf, best_metrics_brf = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             if rng.rand() < 0.4:
#                 c["max_features"] = rng.choice(["sqrt", "log2", None])
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=True, use_sampler=False)
#             if sc > best_score_brf:
#                 best_brf, best_score_brf, best_metrics_brf = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (BRF) | Score={best_score_brf:.4f} | AUC={best_metrics_brf['auc']:.4f} | F1={best_metrics_brf['f1']:.4f} | ACC={best_metrics_brf['acc']:.4f} | Recall={best_metrics_brf['recall']:.4f}")
    
#     # Optimize for BRF with SMOTEENN (separate loop)
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_brf_smoteen, best_score_brf_smoteen, best_metrics_brf_smoteen = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             if rng.rand() < 0.4:
#                 c["max_features"] = rng.choice(["sqrt", "log2", None])
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=True, use_sampler=True)
#             if sc > best_score_brf_smoteen:
#                 best_brf_smoteen, best_score_brf_smoteen, best_metrics_brf_smoteen = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (BRF+SMOTEENN) | Score={best_score_brf_smoteen:.4f} | AUC={best_metrics_brf_smoteen['auc']:.4f} | F1={best_metrics_brf_smoteen['f1']:.4f} | ACC={best_metrics_brf_smoteen['acc']:.4f} | Recall={best_metrics_brf_smoteen['recall']:.4f}")

#     return (best_rf, best_metrics_rf), (best_brf, best_metrics_brf), (best_brf_smoteen, best_metrics_brf_smoteen)

# # اجرای بهینه‌سازی پارامترها
# (best_params_rf, best_mets_rf), (best_params_brf, best_mets_brf), (best_params_brf_smoteen, best_mets_brf_smoteen) = aoa_optimize_rf_plusplus(X, y, num_cols, cat_cols, max_iter=15, pop_size=24)

# print("\nBest RF params (AOA++ with SMOTEENN):", best_params_rf)
# print("Best (AOA++ with SMOTEENN) metrics (CV 10x1):", best_mets_rf)

# print("\nBest BRF params (AOA++ without SMOTEENN):", best_params_brf)
# print("Best (AOA++ without SMOTEENN) metrics (CV 10x1):", best_mets_brf)

# print("\nBest BRF params (AOA++ with SMOTEENN):", best_params_brf_smoteen)
# print("Best (AOA++ with SMOTEENN) metrics (CV 10x1):", best_mets_brf_smoteen)

# # ساخت مدل‌ها با پارامتر بهتر و pipeline
# rf_final = RandomForestClassifier(**best_params_rf, random_state=42, n_jobs=-1)
# pipe_final = build_pipeline(rf_final, num_cols, cat_cols, sampler=SMOTEENN(random_state=42))

# brf_final = BalancedRandomForestClassifier(**{k: v for k, v in best_params_brf.items() if k not in ['class_weight', 'bootstrap']}, random_state=42, n_jobs=-1)
# pipe_brf = build_pipeline(brf_final, num_cols, cat_cols, sampler=None)

# brf_final_smoteen = BalancedRandomForestClassifier(**{k: v for k, v in best_params_brf_smoteen.items() if k not in ['class_weight', 'bootstrap']}, random_state=42, n_jobs=-1)
# pipe_brf_smoteen = build_pipeline(brf_final_smoteen, num_cols, cat_cols, sampler=SMOTEENN(random_state=42))

# # تقسیم داده
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# # آموزش مدل‌ها
# pipe_final.fit(X_train, y_train)
# pipe_brf.fit(X_train, y_train)
# pipe_brf_smoteen.fit(X_train, y_train)

# # تابع محاسبه متریک روی train (OOF)
# def calc_metrics(pipe, X, y):
#     proba = cross_val_predict(pipe, X, y, cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
#                              method="predict_proba", n_jobs=-1)[:, 1]
#     preds = (proba >= 0.5).astype(int)
#     return {
#         "accuracy": accuracy_score(y, preds),
#         "roc_auc": roc_auc_score(y, proba),
#         "recall": recall_score(y, preds),
#         "precision": precision_score(y, preds),
#         "f1": f1_score(y, preds)
#     }

# # محاسبه متریک روی داده آموزش (train)
# metrics_train_rf = calc_metrics(pipe_final, X_train, y_train)
# metrics_train_brf = calc_metrics(pipe_brf, X_train, y_train)
# metrics_train_brf_smoteen = calc_metrics(pipe_brf_smoteen, X_train, y_train)

# # تابع محاسبه متریک و خروجی احتمال روی test
# def calc_metrics_test(pipe, X, y):
#     proba = pipe.predict_proba(X)[:, 1]
#     preds = (proba >= 0.5).astype(int)
#     return {
#         "accuracy": accuracy_score(y, preds),
#         "roc_auc": roc_auc_score(y, proba),
#         "recall": recall_score(y, preds),
#         "precision": precision_score(y, preds),
#         "f1": f1_score(y, preds)
#     }, proba

# # محاسبه متریک و احتمال روی داده تست (test)
# metrics_test_rf, proba_test_rf = calc_metrics_test(pipe_final, X_test, y_test)
# metrics_test_brf, proba_test_brf = calc_metrics_test(pipe_brf, X_test, y_test)
# metrics_test_brf_smoteen, proba_test_brf_smoteen = calc_metrics_test(pipe_brf_smoteen, X_test, y_test)

# # چاپ متریک‌ها (دسته‌بندی)
# print("\nFinal RF+SMOTEENN (Train/Oof):")
# for k, v in metrics_train_rf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal RF+SMOTEENN (Test):")
# for k, v in metrics_test_rf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF (Train/Oof):")
# for k, v in metrics_train_brf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF (Test):")
# for k, v in metrics_test_brf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF+SMOTEENN (Train/Oof):")
# for k, v in metrics_train_brf_smoteen.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF+SMOTEENN (Test):")
# for k, v in metrics_test_brf_smoteen.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# # رسم نمودار ROC و Precision-Recall فقط روی داده تست
# plt.figure(figsize=(14, 6))

# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_rf).plot()
# plt.title("ROC Curve RF (Test)")

# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_rf).plot()
# plt.title("Precision-Recall Curve RF (Test)")

# plt.show()

# plt.figure(figsize=(14, 6))

# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_brf).plot()
# plt.title("ROC Curve BRF (Test)")

# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_brf).plot()
# plt.title("Precision-Recall Curve BRF (Test)")

# plt.show()

# plt.figure(figsize=(14, 6))

# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_brf_smoteen).plot()
# plt.title("ROC Curve BRF+SMOTEENN (Test)")

# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_brf_smoteen).plot()
# plt.title("Precision-Recall Curve BRF+SMOTEENN (Test)")

# plt.show()


In [ ]:
# # ===== Block 11: AOA Optimization + Final Modeling + Evaluation (max_iter=20, pop_size=32) =====

# def aoa_optimize_rf(X, y, num_cols, cat_cols, max_iter=20, pop_size=32, seed=42):
#     rng = np.random.RandomState(seed)
#     def sample_candidate():
#         return {
#             "n_estimators": int(rng.randint(150, 601)),
#             "max_depth": rng.choice([None, 3, 5, 7, 10]),
#             "min_samples_split": int(rng.randint(2, 13)),
#             "min_samples_leaf": int(rng.randint(1, 7)),
#             "max_features": rng.choice(['sqrt', 'log2', None]),
#             "bootstrap": bool(rng.randint(0,2)),
#             "class_weight": rng.choice([None, 'balanced'])
#         }
#     def build_rf(p):
#         return RandomForestClassifier(
#             n_estimators=p["n_estimators"],
#             max_depth=p["max_depth"],
#             min_samples_split=p["min_samples_split"],
#             min_samples_leaf=p["min_samples_leaf"],
#             max_features=p["max_features"],
#             bootstrap=p["bootstrap"],
#             class_weight=p["class_weight"],
#             n_jobs=-1,
#             random_state=seed
#         )
#     def build_brf(p):
#         return BalancedRandomForestClassifier(
#             n_estimators=p["n_estimators"],
#             max_depth=p["max_depth"],
#             min_samples_split=p["min_samples_split"],
#             min_samples_leaf=p["min_samples_leaf"],
#             max_features=p["max_features"],
#             random_state=seed,
#             n_jobs=-1
#         )
#     def fitness(p, use_brf=False, use_sampler=False):
#         if use_brf:
#             estimator = build_brf(p)
#             sampler = SMOTEENN(random_state=seed) if use_sampler else None
#             pipe = build_pipeline(estimator, num_cols, cat_cols, sampler=sampler)
#         else:
#             estimator = build_rf(p)
#             pipe = build_pipeline(estimator, num_cols, cat_cols, sampler=SMOTEENN(random_state=seed))
#         cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
#         oof_proba = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
#         oof_pred = (oof_proba >= 0.5).astype(int)
#         auc = roc_auc_score(y, oof_proba)
#         f1 = f1_score(y, oof_pred)
#         recall = recall_score(y, oof_pred)
#         acc = accuracy_score(y, oof_pred)
#         precision = precision_score(y, oof_pred)
#         score = 0.4 * auc + 0.4 * recall + 0.1 * f1 + 0.1 * acc
#         return score, {"auc": auc, "acc": acc, "f1": f1, "recall": recall, "precision": precision}

#     # RF+SMOTEENN
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_rf, best_score_rf, best_metrics_rf = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             if rng.rand() < 0.4:
#                 c["max_features"] = rng.choice(["sqrt", "log2", None])
#             if rng.rand() < 0.3:
#                 c["bootstrap"] = not c["bootstrap"]
#             if rng.rand() < 0.3:
#                 c["class_weight"] = rng.choice([None, "balanced"])
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=False)
#             if sc > best_score_rf:
#                 best_rf, best_score_rf, best_metrics_rf = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (RF+SMOTEENN) | Score={best_score_rf:.4f} | AUC={best_metrics_rf['auc']:.4f} | F1={best_metrics_rf['f1']:.4f} | ACC={best_metrics_rf['acc']:.4f} | Recall={best_metrics_rf['recall']:.4f}")

#     # BRF بدون SMOTEENN
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_brf, best_score_brf, best_metrics_brf = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             if rng.rand() < 0.4:
#                 c["max_features"] = rng.choice(["sqrt", "log2", None])
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=True, use_sampler=False)
#             if sc > best_score_brf:
#                 best_brf, best_score_brf, best_metrics_brf = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (BRF) | Score={best_score_brf:.4f} | AUC={best_metrics_brf['auc']:.4f} | F1={best_metrics_brf['f1']:.4f} | ACC={best_metrics_brf['acc']:.4f} | Recall={best_metrics_brf['recall']:.4f}")

#     # BRF+SMOTEENN
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_brf_smoteen, best_score_brf_smoteen, best_metrics_brf_smoteen = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             if rng.rand() < 0.4:
#                 c["max_features"] = rng.choice(["sqrt", "log2", None])
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=True, use_sampler=True)
#             if sc > best_score_brf_smoteen:
#                 best_brf_smoteen, best_score_brf_smoteen, best_metrics_brf_smoteen = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (BRF+SMOTEENN) | Score={best_score_brf_smoteen:.4f} | AUC={best_metrics_brf_smoteen['auc']:.4f} | F1={best_metrics_brf_smoteen['f1']:.4f} | ACC={best_metrics_brf_smoteen['acc']:.4f} | Recall={best_metrics_brf_smoteen['recall']:.4f}")

#     return (best_rf, best_metrics_rf), (best_brf, best_metrics_brf), (best_brf_smoteen, best_metrics_brf_smoteen)

# # اجرای بهینه‌سازی پارامترها با AOA
# (best_params_rf, best_mets_rf), (best_params_brf, best_mets_brf), (best_params_brf_smoteen, best_mets_brf_smoteen) = \
#     aoa_optimize_rf(X, y, num_cols, cat_cols, max_iter=20, pop_size=32)

# print("\nBest RF params (AOA):", best_params_rf)
# print("Best (AOA) metrics (CV 10x1):", best_mets_rf)

# print("\nBest BRF params (AOA):", best_params_brf)
# print("Best (AOA) metrics (CV 10x1):", best_mets_brf)

# print("\nBest BRF+SMOTEENN params (AOA):", best_params_brf_smoteen)
# print("Best (AOA with SMOTEENN) metrics (CV 10x1):", best_mets_brf_smoteen)

# # ===== Build final pipelines =====
# rf_final = RandomForestClassifier(**best_params_rf, random_state=42, n_jobs=-1)
# pipe_final = build_pipeline(rf_final, num_cols, cat_cols, sampler=SMOTEENN(random_state=42))

# brf_final = BalancedRandomForestClassifier(**{k: v for k, v in best_params_brf.items() if k not in ['class_weight', 'bootstrap']}, random_state=42, n_jobs=-1)
# pipe_brf = build_pipeline(brf_final, num_cols, cat_cols, sampler=None)

# brf_final_smoteen = BalancedRandomForestClassifier(**{k: v for k, v in best_params_brf_smoteen.items() if k not in ['class_weight', 'bootstrap']}, random_state=42, n_jobs=-1)
# pipe_brf_smoteen = build_pipeline(brf_final_smoteen, num_cols, cat_cols, sampler=SMOTEENN(random_state=42))

# # ===== Train/Test split =====
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
# pipe_final.fit(X_train, y_train)
# pipe_brf.fit(X_train, y_train)
# pipe_brf_smoteen.fit(X_train, y_train)

# # ===== OOF metrics on X_train =====
# def calc_metrics(pipe, X, y):
#     proba = cross_val_predict(pipe, X, y, cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
#                              method="predict_proba", n_jobs=-1)[:, 1]
#     preds = (proba >= 0.5).astype(int)
#     return {
#         "accuracy": accuracy_score(y, preds),
#         "roc_auc": roc_auc_score(y, proba),
#         "recall": recall_score(y, preds),
#         "precision": precision_score(y, preds),
#         "f1": f1_score(y, preds)
#     }, proba

# metrics_train_rf, proba_train_rf = calc_metrics(pipe_final, X_train, y_train)
# metrics_train_brf, proba_train_brf = calc_metrics(pipe_brf, X_train, y_train)
# metrics_train_brf_smoteen, proba_train_brf_sm = calc_metrics(pipe_brf_smoteen, X_train, y_train)

# # ===== Test metrics on X_test =====
# def calc_metrics_test(pipe, X, y):
#     proba = pipe.predict_proba(X)[:, 1]
#     preds = (proba >= 0.5).astype(int)
#     return {
#         "accuracy": accuracy_score(y, preds),
#         "roc_auc": roc_auc_score(y, proba),
#         "recall": recall_score(y, preds),
#         "precision": precision_score(y, preds),
#         "f1": f1_score(y, preds)
#     }, proba

# metrics_test_rf, proba_test_rf = calc_metrics_test(pipe_final, X_test, y_test)
# metrics_test_brf, proba_test_brf = calc_metrics_test(pipe_brf, X_test, y_test)
# metrics_test_brf_smoteen, proba_test_brf_sm = calc_metrics_test(pipe_brf_smoteen, X_test, y_test)

# # ===== Print metrics (Train OOF) =====
# print("\nFinal RF+SMOTEENN (Train/OOF):")
# for k, v in metrics_train_rf.items(): print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF (Train/OOF):")
# for k, v in metrics_train_brf.items(): print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF+SMOTEENN (Train/OOF):")
# for k, v in metrics_train_brf_smoteen.items(): print(f"{k.capitalize()}: {v:.3f}")

# # ===== Print metrics (Test) =====
# print("\nFinal RF+SMOTEENN (Test):")
# for k, v in metrics_test_rf.items(): print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF (Test):")
# for k, v in metrics_test_brf.items(): print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF+SMOTEENN (Test):")
# for k, v in metrics_test_brf_smoteen.items(): print(f"{k.capitalize()}: {v:.3f}")

# # ===== Plots: ROC and PR for Test (three models) =====
# plt.figure(figsize=(14, 6))
# plt.subplot(1, 2, 1); RocCurveDisplay.from_predictions(y_test, proba_test_rf).plot(); plt.title("ROC - RF (Test)")
# plt.subplot(1, 2, 2); PrecisionRecallDisplay.from_predictions(y_test, proba_test_rf).plot(); plt.title("PR - RF (Test)")
# plt.tight_layout(); plt.show()

# plt.figure(figsize=(14, 6))
# plt.subplot(1, 2, 1); RocCurveDisplay.from_predictions(y_test, proba_test_brf).plot(); plt.title("ROC - BRF (Test)")
# plt.subplot(1, 2, 2); PrecisionRecallDisplay.from_predictions(y_test, proba_test_brf).plot(); plt.title("PR - BRF (Test)")
# plt.tight_layout(); plt.show()

# plt.figure(figsize=(14, 6))
# plt.subplot(1, 2, 1); RocCurveDisplay.from_predictions(y_test, proba_test_brf_sm).plot(); plt.title("ROC - BRF+SMOTEENN (Test)")
# plt.subplot(1, 2, 2); PrecisionRecallDisplay.from_predictions(y_test, proba_test_brf_sm).plot(); plt.title("PR - BRF+SMOTEENN (Test)")
# plt.tight_layout(); plt.show()

# # ===== Optional: ROC/PR for Train (OOF) =====
# # اگر می‌خواهی نمودارهای OOF (Train) هم رسم شود، این بلوک را فعال کن:
# # plt.figure(figsize=(14, 6))
# # plt.subplot(1, 2, 1); RocCurveDisplay.from_predictions(y_train, proba_train_rf).plot(); plt.title("ROC - RF (Train OOF)")
# # plt.subplot(1, 2, 2); PrecisionRecallDisplay.from_predictions(y_train, proba_train_rf).plot(); plt.title("PR - RF (Train OOF)")
# # plt.tight_layout(); plt.show()
# #
# # plt.figure(figsize=(14, 6))
# # plt.subplot(1, 2, 1); RocCurveDisplay.from_predictions(y_train, proba_train_brf).plot(); plt.title("ROC - BRF (Train OOF)")
# # plt.subplot(1, 2, 2); PrecisionRecallDisplay.from_predictions(y_train, proba_train_brf).plot(); plt.title("PR - BRF (Train OOF)")
# # plt.tight_layout(); plt.show()
# #
# # plt.figure(figsize=(14, 6))
# # plt.subplot(1, 2, 1); RocCurveDisplay.from_predictions(y_train, proba_train_brf_sm).plot(); plt.title("ROC - BRF+SMOTEENN (Train OOF)")
# # plt.subplot(1, 2, 2); PrecisionRecallDisplay.from_predictions(y_train, proba_train_brf_sm).plot(); plt.title("PR - BRF+SMOTEENN (Train OOF)")
# # plt.tight_layout(); plt.show()


In [ ]:
# # ===== Block 12: AOA++ Optimization + Final Modeling + Evaluation (max_iter=20, pop_size=32) =====

# def aoa_optimize_rf_plusplus(X, y, num_cols, cat_cols, max_iter=20, pop_size=32, seed=42):
#     rng = np.random.RandomState(seed)
#     def sample_candidate():
#         return {
#             "n_estimators": int(rng.randint(150, 601)),
#             "max_depth": rng.choice([None, 3, 5, 7, 10]),
#             "min_samples_split": int(rng.randint(2, 13)),
#             "min_samples_leaf": int(rng.randint(1, 7)),
#             "max_features": rng.choice(['sqrt', 'log2', None]),
#             "bootstrap": bool(rng.randint(0,2)),
#             "class_weight": rng.choice([None, 'balanced'])
#         }
#     def build_rf(p):
#         return RandomForestClassifier(
#             n_estimators=p["n_estimators"],
#             max_depth=p["max_depth"],
#             min_samples_split=p["min_samples_split"],
#             min_samples_leaf=p["min_samples_leaf"],
#             max_features=p["max_features"],
#             bootstrap=p["bootstrap"],
#             class_weight=p["class_weight"],
#             n_jobs=-1,
#             random_state=seed
#         )
#     def build_brf(p):
#         # class_weight و bootstrap در BRF پشتیبانی نمی‌شود
#         return BalancedRandomForestClassifier(
#             n_estimators=p["n_estimators"],
#             max_depth=p["max_depth"],
#             min_samples_split=p["min_samples_split"],
#             min_samples_leaf=p["min_samples_leaf"],
#             max_features=p["max_features"],
#             random_state=seed,
#             n_jobs=-1
#         )
#     def fitness(p, use_brf=False, use_sampler=False):
#         if use_brf:
#             estimator = build_brf(p)
#             sampler = SMOTEENN(random_state=seed) if use_sampler else None
#             pipe = build_pipeline(estimator, num_cols, cat_cols, sampler=sampler)
#         else:
#             estimator = build_rf(p)
#             pipe = build_pipeline(estimator, num_cols, cat_cols, sampler=SMOTEENN(random_state=seed))
#         cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
#         oof_proba = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
#         oof_pred = (oof_proba >= 0.5).astype(int)
#         auc = roc_auc_score(y, oof_proba)
#         f1 = f1_score(y, oof_pred)
#         recall = recall_score(y, oof_pred)
#         acc = accuracy_score(y, oof_pred)
#         precision = precision_score(y, oof_pred)
#         score = 0.4 * auc + 0.4 * recall + 0.1 * f1 + 0.1 * acc
#         return score, {"auc": auc, "acc": acc, "f1": f1, "recall": recall, "precision": precision}

#     # RF+SMOTEENN
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_rf, best_score_rf, best_metrics_rf = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             if rng.rand() < 0.4:
#                 c["max_features"] = rng.choice(["sqrt", "log2", None])
#             if rng.rand() < 0.3:
#                 c["bootstrap"] = not c["bootstrap"]
#             if rng.rand() < 0.3:
#                 c["class_weight"] = rng.choice([None, "balanced"])
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=False)
#             if sc > best_score_rf:
#                 best_rf, best_score_rf, best_metrics_rf = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (RF+SMOTEENN) | Score={best_score_rf:.4f} | AUC={best_metrics_rf['auc']:.4f} | F1={best_metrics_rf['f1']:.4f} | ACC={best_metrics_rf['acc']:.4f} | Recall={best_metrics_rf['recall']:.4f}")

#     # BRF بدون SMOTEENN
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_brf, best_score_brf, best_metrics_brf = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             if rng.rand() < 0.4:
#                 c["max_features"] = rng.choice(["sqrt", "log2", None])
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=True, use_sampler=False)
#             if sc > best_score_brf:
#                 best_brf, best_score_brf, best_metrics_brf = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (BRF) | Score={best_score_brf:.4f} | AUC={best_metrics_brf['auc']:.4f} | F1={best_metrics_brf['f1']:.4f} | ACC={best_metrics_brf['acc']:.4f} | Recall={best_metrics_brf['recall']:.4f}")

#     # BRF+SMOTEENN
#     population = [sample_candidate() for _ in range(pop_size)]
#     best_brf_smoteen, best_score_brf_smoteen, best_metrics_brf_smoteen = None, -np.inf, None
#     for it in range(max_iter):
#         new_pop = []
#         for cand in population:
#             c = cand.copy()
#             if rng.rand() < 0.6:
#                 c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
#             if rng.rand() < 0.4:
#                 c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
#             if rng.rand() < 0.5:
#                 c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
#             if rng.rand() < 0.5:
#                 c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
#             if rng.rand() < 0.4:
#                 c["max_features"] = rng.choice(["sqrt", "log2", None])
#             new_pop.append(c)
#         for cand in new_pop:
#             sc, mets = fitness(cand, use_brf=True, use_sampler=True)
#             if sc > best_score_brf_smoteen:
#                 best_brf_smoteen, best_score_brf_smoteen, best_metrics_brf_smoteen = cand, sc, mets
#         population = new_pop
#         print(f"Iter {it+1}/{max_iter} (BRF+SMOTEENN) | Score={best_score_brf_smoteen:.4f} | AUC={best_metrics_brf_smoteen['auc']:.4f} | F1={best_metrics_brf_smoteen['f1']:.4f} | ACC={best_metrics_brf_smoteen['acc']:.4f} | Recall={best_metrics_brf_smoteen['recall']:.4f}")

#     return (best_rf, best_metrics_rf), (best_brf, best_metrics_brf), (best_brf_smoteen, best_metrics_brf_smoteen)

# # اجرای بهینه‌سازی پارامترها به روش پیشرفته‌تر (AOA++)
# (best_params_rf, best_mets_rf), (best_params_brf, best_mets_brf), (best_params_brf_smoteen, best_mets_brf_smoteen) = \
#     aoa_optimize_rf_plusplus(X, y, num_cols, cat_cols, max_iter=20, pop_size=32)

# print("\nBest RF params (AOA++):", best_params_rf)
# print("Best (AOA++) metrics (CV 10x1):", best_mets_rf)

# print("\nBest BRF params (AOA++):", best_params_brf)
# print("Best (AOA++) metrics (CV 10x1):", best_mets_brf)

# print("\nBest BRF+SMOTEENN params (AOA++):", best_params_brf_smoteen)
# print("Best (AOA++ with SMOTEENN) metrics (CV 10x1):", best_mets_brf_smoteen)

# # ساخت مدل‌ها با پارامتر بهتر و pipeline (بدون هیچ تغییر دیگری)
# rf_final = RandomForestClassifier(**best_params_rf, random_state=42, n_jobs=-1)
# pipe_final = build_pipeline(rf_final, num_cols, cat_cols, sampler=SMOTEENN(random_state=42))

# brf_final = BalancedRandomForestClassifier(**{k: v for k, v in best_params_brf.items() if k not in ['class_weight', 'bootstrap']}, random_state=42, n_jobs=-1)
# pipe_brf = build_pipeline(brf_final, num_cols, cat_cols, sampler=None)

# brf_final_smoteen = BalancedRandomForestClassifier(**{k: v for k, v in best_params_brf_smoteen.items() if k not in ['class_weight', 'bootstrap']}, random_state=42, n_jobs=-1)
# pipe_brf_smoteen = build_pipeline(brf_final_smoteen, num_cols, cat_cols, sampler=SMOTEENN(random_state=42))

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
# pipe_final.fit(X_train, y_train)
# pipe_brf.fit(X_train, y_train)
# pipe_brf_smoteen.fit(X_train, y_train)

# def calc_metrics(pipe, X, y):
#     proba = cross_val_predict(pipe, X, y, cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
#                              method="predict_proba", n_jobs=-1)[:, 1]
#     preds = (proba >= 0.5).astype(int)
#     return {
#         "accuracy": accuracy_score(y, preds),
#         "roc_auc": roc_auc_score(y, proba),
#         "recall": recall_score(y, preds),
#         "precision": precision_score(y, preds),
#         "f1": f1_score(y, preds)
#     }

# metrics_train_rf = calc_metrics(pipe_final, X_train, y_train)
# metrics_train_brf = calc_metrics(pipe_brf, X_train, y_train)
# metrics_train_brf_smoteen = calc_metrics(pipe_brf_smoteen, X_train, y_train)

# def calc_metrics_test(pipe, X, y):
#     proba = pipe.predict_proba(X)[:, 1]
#     preds = (proba >= 0.5).astype(int)
#     return {
#         "accuracy": accuracy_score(y, preds),
#         "roc_auc": roc_auc_score(y, proba),
#         "recall": recall_score(y, preds),
#         "precision": precision_score(y, preds),
#         "f1": f1_score(y, preds)
#     }, proba

# metrics_test_rf, proba_test_rf = calc_metrics_test(pipe_final, X_test, y_test)
# metrics_test_brf, proba_test_brf = calc_metrics_test(pipe_brf, X_test, y_test)
# metrics_test_brf_smoteen, proba_test_brf_smoteen = calc_metrics_test(pipe_brf_smoteen, X_test, y_test)

# print("\nFinal RF+SMOTEENN (Train/Oof):")
# for k, v in metrics_train_rf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal RF+SMOTEENN (Test):")
# for k, v in metrics_test_rf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF (Train/Oof):")
# for k, v in metrics_train_brf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF (Test):")
# for k, v in metrics_test_brf.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF+SMOTEENN (Train/Oof):")
# for k, v in metrics_train_brf_smoteen.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# print("\nFinal BRF+SMOTEENN (Test):")
# for k, v in metrics_test_brf_smoteen.items():
#     print(f"{k.capitalize()}: {v:.3f}")

# plt.figure(figsize=(14, 6))
# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_rf).plot()
# plt.title("ROC Curve RF (Test)")
# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_rf).plot()
# plt.title("Precision-Recall Curve RF (Test)")
# plt.show()

# plt.figure(figsize=(14, 6))
# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_brf).plot()
# plt.title("ROC Curve BRF (Test)")
# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_brf).plot()
# plt.title("Precision-Recall Curve BRF (Test)")
# plt.show()

# plt.figure(figsize=(14, 6))
# plt.subplot(1, 2, 1)
# RocCurveDisplay.from_predictions(y_test, proba_test_brf_smoteen).plot()
# plt.title("ROC Curve BRF+SMOTEENN (Test)")
# plt.subplot(1, 2, 2)
# PrecisionRecallDisplay.from_predictions(y_test, proba_test_brf_smoteen).plot()
# plt.title("Precision-Recall Curve BRF+SMOTEENN (Test)")
# plt.show()


# ==============================================================================
# PART 3: FINAL EVALUATION, BASELINES & XAI
# ==============================================================================

In [ ]:
# ===== block 13: AOA/AOA++ optimizer for calibrated BRF (Modified for Timers & Statistical Testing) =====
import time
from sklearn.model_selection import cross_val_score

def build_pipeline(model, num_cols, cat_cols, sampler=None):
    num_pipe = SkPipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler())
    ])
    cat_pipe = SkPipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore"))
    ])
    pre = ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ])
    steps = [("pre", pre)]
    if sampler is not None:
        steps.append(("sampler", sampler))
    steps.append(("model", model))
    return ImbPipeline(steps)

# ===== 1) Threshold search helper =====
def find_best_threshold(y_true, proba, metric="f1"):
    thresholds = np.linspace(0.05, 0.95, 37)
    best_t, best_val = 0.5, -np.inf
    for t in thresholds:
        preds = (proba >= t).astype(int)
        if metric == "f1":
            val = f1_score(y_true, preds)
        elif metric == "balanced_acc":
            from sklearn.metrics import balanced_accuracy_score
            val = balanced_accuracy_score(y_true, preds)
        else:
            val = f1_score(y_true, preds)
        if val > best_val:
            best_val, best_t = val, t
    return best_t, best_val

# ===== 2) Feature selector (compatible with sklearn clone) =====
class FeatureNameSelector(BaseEstimator, TransformerMixin):
    def __init__(self, preprocessor=None, selected_feature_names=None):
        self.preprocessor = preprocessor
        self.selected_feature_names = selected_feature_names if selected_feature_names is not None else []
        self._idxs_ = None

    def fit(self, X, y=None):
        self.preprocessor.fit(X, y)
        all_feats = self.preprocessor.get_feature_names_out()
        name_to_idx = {n: i for i, n in enumerate(all_feats)}
        self._idxs_ = np.array(
            [name_to_idx[n] for n in self.selected_feature_names if n in name_to_idx],
            dtype=int
        )
        return self

    def transform(self, X):
        Xt = self.preprocessor.transform(X)
        if self._idxs_ is None or len(self._idxs_) == 0:
            return Xt
        return Xt[:, self._idxs_]

# ===== 3) Permutation Importance → Top-K features (no sampler) =====
def compute_perm_importance_topk(X, y, num_cols, cat_cols, K=15, seed=42):
    temp_model = BalancedRandomForestClassifier(
        n_estimators=300,
        random_state=seed,
        n_jobs=-1
    )
    temp_pipe = build_pipeline(temp_model, num_cols, cat_cols, sampler=None)

    temp_pipe.fit(X, y)
    pre = temp_pipe.named_steps["pre"]
    feat_names = pre.get_feature_names_out()

    result = permutation_importance(
        temp_pipe, X, y,
        scoring="roc_auc",
        n_repeats=20,
        random_state=seed,
        n_jobs=-1
    )

    features_used = feat_names[:len(result.importances_mean)]
    imp_df = pd.DataFrame({
        "feature": features_used,
        "mean": result.importances_mean,
        "std": result.importances_std
    }).sort_values("mean", ascending=False).reset_index(drop=True)

    plt.figure(figsize=(7, 5))
    sns.barplot(data=imp_df.head(K), x="mean", y="feature", orient="h")
    plt.title(f"Top-{K} Features (Permutation Importance)")
    plt.tight_layout()
    plt.show()

    top_features = imp_df.head(K)["feature"].tolist()
    return top_features, imp_df

# ===== 4) AOA++ helper for BRF + calibration (no sampler) =====
def aoa_optimize_brf_calib(
    X, y, num_cols, cat_cols, selected_feature_names,
    max_iter=10, pop_size=16, seed=42, return_history=True
):
    
    start_time = time.time()
    rng = np.random.RandomState(seed)

    num_pipe = SkPipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler())
    ])
    cat_pipe = SkPipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore"))
    ])
    base_pre = ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ])

    def sample_candidate():
        return {
            "n_estimators": int(rng.randint(150, 601)),
            "max_depth": rng.choice([None, 3, 5, 7, 10]),
            "min_samples_split": int(rng.randint(2, 13)),
            "min_samples_leaf": int(rng.randint(1, 7)),
            "max_features": rng.choice(['sqrt', 'log2', None])
        }

    def build_brf_calibrated(p):
        base = BalancedRandomForestClassifier(
            n_estimators=p["n_estimators"],
            max_depth=p["max_depth"],
            min_samples_split=p["min_samples_split"],
            min_samples_leaf=p["min_samples_leaf"],
            max_features=p["max_features"],
            random_state=seed,
            n_jobs=-1
        )
        return CalibratedClassifierCV(base, method="isotonic", cv=5)

    def build_fs_calib_pipeline(p):
        feat_sel = FeatureNameSelector(
            preprocessor=base_pre,
            selected_feature_names=selected_feature_names
        )
        model = build_brf_calibrated(p)
        return ImbPipeline([
            ("feat_pre", feat_sel),
            ("model", model)
        ])

    def fitness(p):
        pipe = build_fs_calib_pipeline(p)
        cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
        oof_proba = cross_val_predict(
            pipe, X, y,
            cv=cv,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]
        oof_pred = (oof_proba >= 0.5).astype(int)
        auc = roc_auc_score(y, oof_proba)
        rec = recall_score(y, oof_pred)
        f1 = f1_score(y, oof_pred)
        acc = accuracy_score(y, oof_pred)
        sc = 0.4*auc + 0.4*rec + 0.1*f1 + 0.1*acc
        
        
        raw_folds = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
        
        return sc, {"auc": auc, "recall": rec, "f1": f1, "acc": acc}, raw_folds

    population = [sample_candidate() for _ in range(pop_size)]
    best_p, best_score, best_mets, best_folds = None, -np.inf, None, None
    history = []

    for it in range(max_iter):
        new_pop = []
        for cand in population:
            c = cand.copy()
            if rng.rand() < 0.6:
                c["n_estimators"] = int(np.clip(c["n_estimators"] + rng.normal(0, 80), 150, 600))
            if rng.rand() < 0.4:
                c["max_depth"] = rng.choice([None, 3, 5, 7, 10])
            if rng.rand() < 0.5:
                c["min_samples_split"] = int(np.clip(c["min_samples_split"] + rng.randint(-2, 3), 2, 12))
            if rng.rand() < 0.5:
                c["min_samples_leaf"] = int(np.clip(c["min_samples_leaf"] + rng.randint(-1, 2), 1, 6))
            if rng.rand() < 0.4:
                c["max_features"] = rng.choice(['sqrt', 'log2', None])
            new_pop.append(c)

        for cand in new_pop:
            sc, mets, f_scores = fitness(cand)
            if return_history:
                history.append({
                    "iter": it + 1,
                    **cand,
                    **mets,
                    "score": sc
                })
            if sc > best_score:
                best_p, best_score, best_mets, best_folds = cand, sc, mets, f_scores

        population = new_pop
        print(
            f"Iter {it+1}/{max_iter} (BRF-Calib) | "
            f"Score={best_score:.4f} | "
            f"AUC={best_mets['auc']:.4f} | "
            f"F1={best_mets['f1']:.4f} | "
            f"ACC={best_mets['acc']:.4f} | "
            f"Recall={best_mets['recall']:.4f}"
        )

    hist_df = pd.DataFrame(history) if return_history else None
    execution_time = time.time() - start_time
    
    
    print("\n" + "="*50)
    print(f"⏱️ Proposed Calibrated Pipeline Execution Time: {execution_time:.2f} seconds")
    print("="*50)
    
  
    return best_p, best_mets, best_folds, hist_df

In [ ]:
# ===== Block 14: Standard Baselines 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, cross_val_score, GridSearchCV
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score, accuracy_score, RocCurveDisplay
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from skopt import BayesSearchCV  

# ------------------------------------------------------------------------------
# 0. Helper Function for 95% Confidence Interval (Bootstrapping) - Focused on Recall
# ------------------------------------------------------------------------------
def compute_bootstrap_metrics_with_ci(y_true, y_prob, threshold, n_bootstraps=1000, seed=42):
    
    rng = np.random.RandomState(seed)
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    
    boot_aucs = []
    boot_recalls = []
    
    for _ in range(n_bootstraps):
        indices = rng.randint(0, len(y_true), len(y_true))
        if len(np.unique(y_true[indices])) < 2:
            continue
            
        auc = roc_auc_score(y_true[indices], y_prob[indices])
        boot_aucs.append(auc)
        
        preds = (y_prob[indices] >= threshold).astype(int)
        recall = recall_score(y_true[indices], preds)
        boot_recalls.append(recall)
        
    return np.percentile(boot_aucs, 2.5), np.percentile(boot_aucs, 97.5), \
           np.percentile(boot_recalls, 2.5), np.percentile(boot_recalls, 97.5)

# 1. Leak-Safe Data Split
print(">>> Splitting Data (Train/Test) to ensure leak-free evaluation...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

# 2. Global Feature Selection
print("\n>>> Computing Top-15 Features on X_train...")
global_top_features, _ = compute_perm_importance_topk(X_train, y_train, num_cols, cat_cols, K=15, seed=42)

# 3. Pipeline Construction Helper (Calibration as Outer Layer)
def make_baseline_pipeline(base_estimator, selected_feats):
    num_pipe = SkPipeline([("imp", SimpleImputer(strategy="median")), ("scaler", RobustScaler())])
    cat_pipe = SkPipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))])
    preprocessor = ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)])
    feat_sel = FeatureNameSelector(preprocessor=preprocessor, selected_feature_names=selected_feats)
    
    
    calibrated_model = CalibratedClassifierCV(base_estimator, method='isotonic', cv=5)
    return ImbPipeline([("feat_pre", feat_sel), ("model", calibrated_model)])

# 4. Strict HPO Setup (Fixing model__estimator__ path for Calibrated Pipeline)
rf_param_space = {
    'model__estimator__n_estimators': [150, 300, 500],
    'model__estimator__max_depth': [5, 10, None],
    'model__estimator__min_samples_split': [2, 5, 10]
}

raw_rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)
base_pipe_for_hpo = make_baseline_pipeline(raw_rf, global_top_features)
cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("⏳ Fitting GridSearch & Bayesian HPO on Calibrated Pipeline...")
grid_search_baseline = GridSearchCV(base_pipe_for_hpo, param_grid=rf_param_space, cv=cv_inner, scoring='roc_auc', n_jobs=-1)
bayesian_search_baseline = BayesSearchCV(base_pipe_for_hpo, search_spaces=rf_param_space, n_iter=10, cv=cv_inner, scoring='roc_auc', n_jobs=-1, random_state=42)

# 5. Baselines Definition
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
baselines = {
    "LR (Base)": LogisticRegression(class_weight='balanced', max_iter=3000, solver='liblinear', random_state=42),
    "SVM (Base)": SVC(class_weight='balanced', kernel='rbf', probability=True, random_state=42),
    "XGB (Base)": XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss', n_estimators=200, random_state=42, n_jobs=-1),
    "RF (Default)": RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42, n_jobs=-1),
    "RF (GridSearch)": grid_search_baseline,
    "RF (Bayesian Opt)": bayesian_search_baseline
}

# 6. Execution Loop
baseline_results, test_probas, baseline_model_folds = {}, {}, {}
cv_outer = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

print("\n>>> Starting Evaluation Loop with 95% CI for AUC & Recall...")
for name, model in baselines.items():
    print(f"--- Evaluating {name} ---")
    pipe = model if ("GridSearch" in name or "Bayesian" in name) else make_baseline_pipeline(model, global_top_features)
    
    oof_proba = cross_val_predict(pipe, X_train, y_train, cv=cv_outer, method="predict_proba", n_jobs=-1)[:, 1]
    baseline_model_folds[name] = cross_val_score(pipe, X_train, y_train, cv=cv_outer, scoring='roc_auc', n_jobs=-1)
    
    best_t, _ = find_best_threshold(y_train, oof_proba, metric="f1")
    pipe.fit(X_train, y_train)
    proba_test = pipe.predict_proba(X_test)[:, 1]
    preds_test = (proba_test >= best_t).astype(int)
    
    auc_low, auc_high, rec_low, rec_high = compute_bootstrap_metrics_with_ci(y_test, proba_test, best_t)
    
    baseline_results[name] = {
        "AUC (95% CI)": f"{roc_auc_score(y_test, proba_test):.4f} [{auc_low:.4f}, {auc_high:.4f}]",
        "Recall (95% CI)": f"{recall_score(y_test, preds_test):.4f} [{rec_low:.4f}, {rec_high:.4f}]",
        "F1-Score": f"{f1_score(y_test, preds_test):.4f}",
        "Precision": f"{precision_score(y_test, preds_test):.4f}",
        "Acc": f"{accuracy_score(y_test, preds_test):.4f}",
        "Threshold": f"{best_t:.3f}"
    }
    test_probas[name] = proba_test

# 7. Leaderboard & ROC Plot
display(pd.DataFrame(baseline_results).T)
plt.figure(figsize=(10, 6))
ax = plt.gca()
for name, proba in test_probas.items(): RocCurveDisplay.from_predictions(y_test, proba, name=name, ax=ax)
plt.title("ROC Curve - Baselines with 95% CI for Recall"); plt.show()

In [ ]:
# ===== Block 15: Pairwise Statistical Comparison of Baseline Models =====
import scipy.stats as stats
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import itertools

print("="*95)
print("📊 STEP 4: PAIRWISE STATISTICAL SIGNIFICANCE ANALYSIS (BASELINE MODELS ONLY)")
print("="*95)



baseline_names = list(baseline_model_folds.keys())
all_baseline_folds = {name: np.array(baseline_model_folds[name]) for name in baseline_names}


print(f"{'Model Pair':<50} | {'T-Stat':<10} | {'P-Value':<10} | {'Significant'}")
print("-" * 95)

for name1, name2 in itertools.combinations(baseline_names, 2):
    t_stat, p_val = stats.ttest_rel(all_baseline_folds[name1], all_baseline_folds[name2])
    is_sig = "✅ YES" if p_val < 0.05 else "⚠️ NO"
    print(f"{name1 + ' vs ' + name2:<50} | {t_stat:>10.4f} | {p_val:>10.4f} | {is_sig}")


print("-" * 95)
print(f"{'Baseline Model':<40} | {'Std Dev (Stability)'}")
print("-" * 95)
for name in baseline_names:
    print(f"{name:<40} | {np.std(all_baseline_folds[name]):.5f}")
print("="*95 + "\n")



plt.figure(figsize=(12, 6))
plot_data = [all_baseline_folds[name] for name in baseline_names]

box = plt.boxplot(plot_data, patch_artist=True, labels=baseline_names, widths=0.5)


colors = plt.cm.plasma(np.linspace(0, 0.8, len(baseline_names)))
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
    patch.set_edgecolor('black')


for median in box['medians']: median.set_color('#FFB703'); median.set_linewidth(2)
for whisk in box['whiskers']: whisk.set_color('black'); whisk.set_linewidth(1.2)

plt.title("Performance Stability & Statistical Distribution of Baseline Models", fontsize=13, fontweight='bold', pad=15)
plt.ylabel("Validation ROC-AUC Score (10-Fold CV)", fontsize=11, fontweight='bold')
plt.grid(True, linestyle="--", alpha=0.3, axis='y')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.savefig("baseline_statistical_comparison.png", dpi=300, bbox_inches='tight')
plt.show()

print("✅ Baseline pairwise statistical analysis completed successfully.")

In [ ]:
# ===== Block 16: Full Pipelines — PI-based FS + TRUE AOA + Calibration + Evaluation =====
# ===== BRF, RF+SMOTEENN, BRF+SMOTEENN | LEAK-SAFE | Approximate CPU usage control| TEMP-AWARE COOLING =====

import os
import random
import time
import psutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, recall_score, f1_score, accuracy_score,
    precision_score, confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.combine import SMOTEENN
from imblearn.ensemble import BalancedRandomForestClassifier


# -----------------------------------------------------------------------------------------
# Suppress only Permutation Importance figures without modifying the main code
# -----------------------------------------------------------------------------------------
import matplotlib.pyplot as plt

if not hasattr(plt, "_original_show"):
    plt._original_show = plt.show

def show_without_permutation_importance(*args, **kwargs):
    figs_to_close = []

    for fig_num in plt.get_fignums():
        fig = plt.figure(fig_num)

        figure_text = []

        for ax in fig.axes:
            figure_text.append(str(ax.get_title()))
            figure_text.append(str(ax.get_xlabel()))
            figure_text.append(str(ax.get_ylabel()))

            for txt in ax.texts:
                figure_text.append(str(txt.get_text()))

        joined_text = " ".join(figure_text).lower()

        if "permutation importance" in joined_text or "top-15 features" in joined_text:
            figs_to_close.append(fig_num)

    for fig_num in figs_to_close:
        plt.close(fig_num)

    remaining_figs = plt.get_fignums()

    if len(remaining_figs) > 0:
        plt._original_show(*args, **kwargs)

plt.show = show_without_permutation_importance


# -----------------------------------------------------------------------------------------
# 0) Reproducibility + # Approximate CPU usage control
# -----------------------------------------------------------------------------------------
def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_all_seeds(42)

physical_cores = psutil.cpu_count(logical=False) or psutil.cpu_count(logical=True) or 1
allowed_cores = max(1, int(physical_cores * 0.8))

os.environ["OMP_NUM_THREADS"] = str(allowed_cores)
os.environ["MKL_NUM_THREADS"] = str(allowed_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(allowed_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(allowed_cores)

try:
    proc = psutil.Process(os.getpid())
    cpus = proc.cpu_affinity()
    keep_n = max(1, int(len(cpus) * 0.8))
    proc.cpu_affinity(cpus[:keep_n])
except Exception:
    pass

print(f"Allowed CPU cores: {allowed_cores}")


# -----------------------------------------------------------------------------------------
# Temperature-aware cooling
# -----------------------------------------------------------------------------------------
TEMP_THRESHOLD_C = 82.0
TEMP_RESUME_C = 75.0
COOL_DOWN_SECONDS = 5
COOL_EVERY_ITERS = 1
CHECK_TEMP_EVERY_CANDIDATES = 6
MAX_TEMP_WAIT_CYCLES = 30


def get_max_cpu_temp():
    try:
        temps = psutil.sensors_temperatures(fahrenheit=False)
        if not temps:
            return None

        readings = []
        for _, entries in temps.items():
            for e in entries:
                if e.current is not None:
                    readings.append(float(e.current))

        return max(readings) if readings else None
    except Exception:
        return None


def cooling_pause(stage="", force_short_pause=False):
    temp = get_max_cpu_temp()

    if temp is None:
        if force_short_pause and COOL_DOWN_SECONDS > 0:
            time.sleep(COOL_DOWN_SECONDS)
        return

    if temp >= TEMP_THRESHOLD_C:
        print(
            f"🌡️ High CPU temperature detected at {temp:.1f}°C "
            f"during {stage}. Cooling down..."
        )

        wait_cycles = 0
        while temp is not None and temp > TEMP_RESUME_C and wait_cycles < MAX_TEMP_WAIT_CYCLES:
            time.sleep(COOL_DOWN_SECONDS)
            temp = get_max_cpu_temp()
            wait_cycles += 1

        if temp is not None:
            print(f"✅ Temperature after cooling: {temp:.1f}°C")

    elif force_short_pause and COOL_DOWN_SECONDS > 0:
        time.sleep(COOL_DOWN_SECONDS)


# -----------------------------------------------------------------------------------------
# 1) Leak-safe train/test split
# -----------------------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


# -----------------------------------------------------------------------------------------
# 2) Feature selector compatible with raw + transformed feature names
# -----------------------------------------------------------------------------------------
class FeatureNameSelector(BaseEstimator, TransformerMixin):
    def __init__(self, preprocessor=None, selected_feature_names=None):
        self.preprocessor = preprocessor
        self.selected_feature_names = selected_feature_names if selected_feature_names is not None else []

    def fit(self, X, y=None):
        self.preprocessor_ = clone(self.preprocessor)
        self.preprocessor_.fit(X, y)

        all_feats = np.asarray(self.preprocessor_.get_feature_names_out())
        selected = set(map(str, self.selected_feature_names))

        idxs = []

        for i, fname in enumerate(all_feats):
            fname = str(fname)
            clean = fname.split("__", 1)[-1]

            keep = False

            if fname in selected or clean in selected:
                keep = True

            for s in selected:
                if clean == s or clean.startswith(s + "_"):
                    keep = True
                    break

            if keep:
                idxs.append(i)

        self.feature_names_out_ = all_feats
        self.selected_indices_ = np.array(idxs, dtype=int)

        if len(self.selected_indices_) == 0:
            print("⚠️ No exact selected feature matched. Returning all processed features.")

        return self

    def transform(self, X):
        Xt = self.preprocessor_.transform(X)

        if self.selected_indices_ is None or len(self.selected_indices_) == 0:
            return Xt

        return Xt[:, self.selected_indices_]


# -----------------------------------------------------------------------------------------
# 3) Select Top-K features using Permutation Importance on the training set only
# -----------------------------------------------------------------------------------------
K = 15

top_features, imp_df = compute_perm_importance_topk(
    X_train, y_train, num_cols, cat_cols, K=K, seed=42
)

print(f"Selected Top-{K} features (train only):", top_features)


# -----------------------------------------------------------------------------------------
# 4) Preprocessing + calibrated pipeline
# -----------------------------------------------------------------------------------------
num_pipe = SkPipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

cat_pipe = SkPipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

base_pre = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])


def make_calibrated_classifier(base_estimator):
    try:
        return CalibratedClassifierCV(
            estimator=base_estimator,
            method="isotonic",
            cv=5
        )
    except TypeError:
        return CalibratedClassifierCV(
            base_estimator=base_estimator,
            method="isotonic",
            cv=5
        )


def make_fs_calibrated_pipeline(base_estimator, selected_feature_names, sampler=None):
    feat_sel = FeatureNameSelector(
        preprocessor=base_pre,
        selected_feature_names=selected_feature_names
    )

    model_cal = make_calibrated_classifier(base_estimator)

    steps = [("feat_pre", feat_sel)]

    if sampler is not None:
        steps.append(("sampler", sampler))

    steps.append(("model", model_cal))

    return ImbPipeline(steps)


# -----------------------------------------------------------------------------------------
# 5) AOA search space
# Xi is optimized in normalized continuous space [0, 1]^5
# Then Xi is decoded to actual RF/BRF hyperparameters.
# -----------------------------------------------------------------------------------------
MAX_DEPTH_CHOICES = [None, 3, 5, 7, 10]
MAX_FEATURES_CHOICES = ["sqrt", "log2", None]


def _decode_int(z, low, high):
    z = float(np.clip(z, 0, 1))
    return int(round(low + z * (high - low)))


def _decode_choice(z, choices):
    z = float(np.clip(z, 0, 1))
    idx = int(round(z * (len(choices) - 1)))
    idx = int(np.clip(idx, 0, len(choices) - 1))
    return choices[idx]


def decode_candidate_vector(x_vec):
    x_vec = np.clip(np.asarray(x_vec, dtype=float), 0, 1)

    return {
        "n_estimators": _decode_int(x_vec[0], 150, 600),
        "max_depth": _decode_choice(x_vec[1], MAX_DEPTH_CHOICES),
        "min_samples_split": _decode_int(x_vec[2], 2, 12),
        "min_samples_leaf": _decode_int(x_vec[3], 1, 6),
        "max_features": _decode_choice(x_vec[4], MAX_FEATURES_CHOICES),
    }


# -----------------------------------------------------------------------------------------
# 6) Estimator builders with 80% CPU
# -----------------------------------------------------------------------------------------
def build_brf(p):
    return BalancedRandomForestClassifier(
        n_estimators=p["n_estimators"],
        max_depth=p["max_depth"],
        min_samples_split=p["min_samples_split"],
        min_samples_leaf=p["min_samples_leaf"],
        max_features=p["max_features"],
        random_state=42,
        n_jobs=allowed_cores
    )


def build_rf(p):
    return RandomForestClassifier(
        n_estimators=p["n_estimators"],
        max_depth=p["max_depth"],
        min_samples_split=p["min_samples_split"],
        min_samples_leaf=p["min_samples_leaf"],
        max_features=p["max_features"],
        random_state=42,
        n_jobs=allowed_cores
    )


# -----------------------------------------------------------------------------------------
# 7) AOA-style optimizer based on Algorithm 1
# alpha is fixed at 2.0.
# Fitness is evaluated on the validation set.
# Since the objective score must be maximized, fitness is defined as -score for minimization.
# -----------------------------------------------------------------------------------------
def aoa_optimize(
    build_estimator_fn,
    selected_feature_names,
    use_smoteen=False,
    max_iter=15,
    pop_size=24,
    seed=42,
    label="GEN",
    X_train=None,
    y_train=None,
    alpha=2.0
):
    rng = np.random.RandomState(seed)
    sampler = SMOTEENN(random_state=seed) if use_smoteen else None

    start_time = time.time()


    X_aoa_train, X_aoa_val, y_aoa_train, y_aoa_val = train_test_split(
        X_train,
        y_train,
        test_size=0.2,
        stratify=y_train,
        random_state=seed
    )

    def fitness(x_vec):
        params = decode_candidate_vector(x_vec)

        est = build_estimator_fn(params)

        pipe = make_fs_calibrated_pipeline(
            est,
            selected_feature_names,
            sampler=sampler
        )

        pipe.fit(X_aoa_train, y_aoa_train)

        val_proba = pipe.predict_proba(X_aoa_val)[:, 1]
        val_pred = (val_proba >= 0.5).astype(int)

        auc = roc_auc_score(y_aoa_val, val_proba)
        rec = recall_score(y_aoa_val, val_pred, zero_division=0)
        f1 = f1_score(y_aoa_val, val_pred, zero_division=0)
        acc = accuracy_score(y_aoa_val, val_pred)

        score = 0.4 * auc + 0.4 * rec + 0.1 * f1 + 0.1 * acc

        # AOA minimizes fitness
        fit_value = -score

        return fit_value, {
            "auc": auc,
            "recall": rec,
            "f1": f1,
            "acc": acc,
            "score": score,
            **params
        }

    # Initialize population Xi for i = 1 to N with random hyperparameters
    dim = 5
    population = rng.uniform(0, 1, size=(pop_size, dim))

    # Xbest ← initial best solution with lowest fitness
    best_x = None
    best_fit = np.inf
    best_mets = None
    history = []

    for i in range(pop_size):
        fit_i, mets_i = fitness(population[i])

        history.append({
            "iter": 0,
            "individual": i + 1,
            "alpha": alpha,
            "fitness": fit_i,
            **mets_i
        })

        if fit_i < best_fit:
            best_x = population[i].copy()
            best_fit = fit_i
            best_mets = mets_i.copy()

        if (i + 1) % CHECK_TEMP_EVERY_CANDIDATES == 0:
            cooling_pause(
                stage=f"{label} initial population {i+1}/{pop_size}",
                force_short_pause=False
            )

    print(
        f"[{label}-Calib] Initial Best | "
        f"Score={best_mets['score']:.4f} | "
        f"AUC={best_mets['auc']:.4f} | "
        f"F1={best_mets['f1']:.4f} | "
        f"ACC={best_mets['acc']:.4f} | "
        f"Recall={best_mets['recall']:.4f}"
    )

    # for iteration t = 1 to T
    for t in range(1, max_iter + 1):

        # Compute XM ← mean of all Xi in population
        XM = np.mean(population, axis=0)

        new_population = []

        # for each individual Xi in population
        for i in range(pop_size):
            Xi = population[i].copy()

            # rand ← uniform random number in [0, 1]
            rand = rng.uniform(0, 1)

            # X'i ← Xbest + α · (rand − 0.5) · (XM − Xi)
            Xi_prime = best_x + alpha * (rand - 0.5) * (XM - Xi)

            # Keep vector inside valid normalized range
            Xi_prime = np.clip(Xi_prime, 0, 1)

            # Evaluate fitness f(X'i)
            fit_prime, mets_prime = fitness(Xi_prime)

            history.append({
                "iter": t,
                "individual": i + 1,
                "alpha": alpha,
                "fitness": fit_prime,
                **mets_prime
            })

            # if f(X'i) < f(Xbest), then Xbest ← X'i
            if fit_prime < best_fit:
                best_x = Xi_prime.copy()
                best_fit = fit_prime
                best_mets = mets_prime.copy()

            # Update position
            new_population.append(Xi_prime)

            if (i + 1) % CHECK_TEMP_EVERY_CANDIDATES == 0:
                cooling_pause(
                    stage=f"{label} iter {t}, candidate {i+1}/{pop_size}",
                    force_short_pause=False
                )

        population = np.array(new_population)

        print(
            f"Iter {t}/{max_iter} ({label}-Calib) | "
            f"Score={best_mets['score']:.4f} | "
            f"AUC={best_mets['auc']:.4f} | "
            f"F1={best_mets['f1']:.4f} | "
            f"ACC={best_mets['acc']:.4f} | "
            f"Recall={best_mets['recall']:.4f}"
        )

        cooling_pause(
            stage=f"{label} iteration {t}",
            force_short_pause=(t % COOL_EVERY_ITERS == 0)
        )

    duration = time.time() - start_time

    best_params = decode_candidate_vector(best_x)
    hist_df = pd.DataFrame(history)

    print(f"⏱️ Optimization for [{label}] completed in {duration:.2f} seconds.\n")

    return best_params, best_mets, hist_df


# -----------------------------------------------------------------------------------------
# 8) Run TRUE AOA for all three models
# -----------------------------------------------------------------------------------------
print(">>> Running TRUE AOA Optimization...")

best_params_brf, best_mets_brf, hist_brf = aoa_optimize(
    build_brf,
    top_features,
    use_smoteen=False,
    max_iter=15,
    pop_size=24,
    seed=42,
    label="BRF",
    X_train=X_train,
    y_train=y_train,
    alpha=2.0
)

print("\nBest params (BRF):", best_params_brf)
print("Best validation metrics (BRF):", best_mets_brf)
display(
    hist_brf.groupby("iter")["score"].max()
    .reset_index()
    .rename(columns={"score": "best_score"})
    .tail()
)

best_params_rf_sm, best_mets_rf_sm, hist_rf_sm = aoa_optimize(
    build_rf,
    top_features,
    use_smoteen=True,
    max_iter=15,
    pop_size=24,
    seed=42,
    label="RF+SMOTEENN",
    X_train=X_train,
    y_train=y_train,
    alpha=2.0
)

print("\nBest params (RF+SMOTEENN):", best_params_rf_sm)
print("Best validation metrics (RF+SMOTEENN):", best_mets_rf_sm)
display(
    hist_rf_sm.groupby("iter")["score"].max()
    .reset_index()
    .rename(columns={"score": "best_score"})
    .tail()
)

best_params_brf_sm, best_mets_brf_sm, hist_brf_sm = aoa_optimize(
    build_brf,
    top_features,
    use_smoteen=True,
    max_iter=15,
    pop_size=24,
    seed=42,
    label="BRF+SMOTEENN",
    X_train=X_train,
    y_train=y_train,
    alpha=2.0
)

print("\nBest params (BRF+SMOTEENN):", best_params_brf_sm)
print("Best validation metrics (BRF+SMOTEENN):", best_mets_brf_sm)
display(
    hist_brf_sm.groupby("iter")["score"].max()
    .reset_index()
    .rename(columns={"score": "best_score"})
    .tail()
)


# -----------------------------------------------------------------------------------------
# 9) Final calibrated pipelines
# -----------------------------------------------------------------------------------------
pipe_brf_fs_calib = make_fs_calibrated_pipeline(
    build_brf(best_params_brf),
    top_features,
    sampler=None
)

pipe_rf_sm_fs_calib = make_fs_calibrated_pipeline(
    build_rf(best_params_rf_sm),
    top_features,
    sampler=SMOTEENN(random_state=42)
)

pipe_brf_sm_fs_calib = make_fs_calibrated_pipeline(
    build_brf(best_params_brf_sm),
    top_features,
    sampler=SMOTEENN(random_state=42)
)


# -----------------------------------------------------------------------------------------
# 10) Threshold optimization
# -----------------------------------------------------------------------------------------
def find_best_threshold(y_true, y_proba, metric="f1"):
    thresholds = np.linspace(0, 1, 200)
    best_thr = 0.5
    best_score = -1

    for thr in thresholds:
        preds = (y_proba >= thr).astype(int)

        if metric == "f1":
            score = f1_score(y_true, preds, zero_division=0)
        elif metric == "recall":
            score = recall_score(y_true, preds, zero_division=0)
        elif metric == "precision":
            score = precision_score(y_true, preds, zero_division=0)
        else:
            score = f1_score(y_true, preds, zero_division=0)

        if score > best_score:
            best_score = score
            best_thr = thr

    return best_thr, best_score


# -----------------------------------------------------------------------------------------
# 11) Evaluation helpers
# -----------------------------------------------------------------------------------------
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


def oof_block(pipe, X_tr, y_tr):
    # n_jobs=1 avoids nested parallel overload.
    # RF/BRF internally use allowed_cores.
    proba_tr = cross_val_predict(
        pipe,
        X_tr,
        y_tr,
        cv=cv,
        method="predict_proba",
        n_jobs=1
    )[:, 1]

    pred_tr_05 = (proba_tr >= 0.5).astype(int)

    t_opt, _ = find_best_threshold(
        y_tr,
        proba_tr,
        metric="f1"
    )

    pred_tr_opt = (proba_tr >= t_opt).astype(int)

    mets_tr_05 = {
        "accuracy": accuracy_score(y_tr, pred_tr_05),
        "roc_auc": roc_auc_score(y_tr, proba_tr),
        "recall": recall_score(y_tr, pred_tr_05, zero_division=0),
        "precision": precision_score(y_tr, pred_tr_05, zero_division=0),
        "f1": f1_score(y_tr, pred_tr_05, zero_division=0),
    }

    mets_tr_opt = {
        "accuracy": accuracy_score(y_tr, pred_tr_opt),
        "roc_auc": roc_auc_score(y_tr, proba_tr),
        "recall": recall_score(y_tr, pred_tr_opt, zero_division=0),
        "precision": precision_score(y_tr, pred_tr_opt, zero_division=0),
        "f1": f1_score(y_tr, pred_tr_opt, zero_division=0),
        "threshold": t_opt,
    }

    return proba_tr, mets_tr_05, mets_tr_opt, t_opt


def test_block(pipe, X_te, y_te, t_opt):
    proba_te = pipe.predict_proba(X_te)[:, 1]

    pred_te_05 = (proba_te >= 0.5).astype(int)
    pred_te_opt = (proba_te >= t_opt).astype(int)

    mets_te_05 = {
        "accuracy": accuracy_score(y_te, pred_te_05),
        "roc_auc": roc_auc_score(y_te, proba_te),
        "recall": recall_score(y_te, pred_te_05, zero_division=0),
        "precision": precision_score(y_te, pred_te_05, zero_division=0),
        "f1": f1_score(y_te, pred_te_05, zero_division=0),
    }

    mets_te_opt = {
        "accuracy": accuracy_score(y_te, pred_te_opt),
        "roc_auc": roc_auc_score(y_te, proba_te),
        "recall": recall_score(y_te, pred_te_opt, zero_division=0),
        "precision": precision_score(y_te, pred_te_opt, zero_division=0),
        "f1": f1_score(y_te, pred_te_opt, zero_division=0),
        "threshold": t_opt,
    }

    return proba_te, mets_te_05, mets_te_opt


def full_eval(pipe, name):
    proba_tr, tr05, tropt, t_opt = oof_block(
        pipe,
        X_train,
        y_train
    )

    pipe.fit(X_train, y_train)

    cooling_pause(
        stage=f"{name} final fit",
        force_short_pause=True
    )

    proba_te, te05, teopt = test_block(
        pipe,
        X_test,
        y_test,
        t_opt
    )

    print(f"\n=== {name} Calibrated with PI-TopK ===")
    print("Train (OOF) @0.50:", {k: round(v, 4) for k, v in tr05.items()})
    print("Train (OOF) @opt :", {k: round(v, 4) for k, v in tropt.items()})
    print("Test @0.50:", {k: round(v, 4) for k, v in te05.items()})
    print("Test @opt :", {k: round(v, 4) for k, v in teopt.items()})

    # ROC & PR - Train
    plt.figure(figsize=(14, 6))

    plt.subplot(1, 2, 1)
    RocCurveDisplay.from_predictions(
        y_train,
        proba_tr,
        ax=plt.gca()
    )
    plt.title(f"ROC - Train (OOF) [{name}]")

    plt.subplot(1, 2, 2)
    PrecisionRecallDisplay.from_predictions(
        y_train,
        proba_tr,
        ax=plt.gca()
    )
    plt.title(f"PR - Train (OOF) [{name}]")

    plt.tight_layout()
    plt.show()

    # ROC & PR - Test
    plt.figure(figsize=(14, 6))

    plt.subplot(1, 2, 1)
    RocCurveDisplay.from_predictions(
        y_test,
        proba_te,
        ax=plt.gca()
    )
    plt.title(f"ROC - Test [{name}]")

    plt.subplot(1, 2, 2)
    PrecisionRecallDisplay.from_predictions(
        y_test,
        proba_te,
        ax=plt.gca()
    )
    plt.title(f"PR - Test [{name}]")

    plt.tight_layout()
    plt.show()

    # Confusion @0.5
    pred_tr_05 = (proba_tr >= 0.5).astype(int)
    pred_te_05 = (proba_te >= 0.5).astype(int)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    sns.heatmap(
        confusion_matrix(y_train, pred_tr_05),
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["Actual 0", "Actual 1"]
    )
    plt.title(f"Confusion Matrix - Train (OOF @0.5) [{name}]")

    plt.subplot(1, 2, 2)
    sns.heatmap(
        confusion_matrix(y_test, pred_te_05),
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["Actual 0", "Actual 1"]
    )
    plt.title(f"Confusion Matrix - Test (@0.5) [{name}]")

    plt.tight_layout()
    plt.show()

    # Confusion @opt
    pred_tr_opt = (proba_tr >= t_opt).astype(int)
    pred_te_opt = (proba_te >= t_opt).astype(int)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    sns.heatmap(
        confusion_matrix(y_train, pred_tr_opt),
        annot=True,
        fmt="d",
        cmap="Greens",
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["Actual 0", "Actual 1"]
    )
    plt.title(f"Confusion Matrix - Train (OOF @opt={t_opt:.2f}) [{name}]")

    plt.subplot(1, 2, 2)
    sns.heatmap(
        confusion_matrix(y_test, pred_te_opt),
        annot=True,
        fmt="d",
        cmap="Greens",
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["Actual 0", "Actual 1"]
    )
    plt.title(f"Confusion Matrix - Test (@opt={t_opt:.2f}) [{name}]")

    plt.tight_layout()
    plt.show()

    return dict(
        train_05=tr05,
        train_opt=tropt,
        test_05=te05,
        test_opt=teopt
    )


# -----------------------------------------------------------------------------------------
# 12) Final evaluation
# -----------------------------------------------------------------------------------------
res_brf = full_eval(
    pipe_brf_fs_calib,
    "BRF"
)

cooling_pause(
    stage="Between BRF and RF+SMOTEENN",
    force_short_pause=True
)

res_rf_sm = full_eval(
    pipe_rf_sm_fs_calib,
    "RF+SMOTEENN"
)

cooling_pause(
    stage="Between RF+SMOTEENN and BRF+SMOTEENN",
    force_short_pause=True
)

res_brf_sm = full_eval(
    pipe_brf_sm_fs_calib,
    "BRF+SMOTEENN"
)

In [ ]:
# ===== Block 17: Statistical Analysis for AOA-Style Optimized Models =====
# ===== CPU-80% Compatible + Temperature-Aware Cooling =====

import itertools
import scipy.stats as stats
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("=" * 95)
print(" STEP 4: Statistical Comparison of AOA-Style Optimized Models")
print("=" * 95)

# -----------------------------------------------------------------------------------------
# 0) CPU / cooling compatibility
# -----------------------------------------------------------------------------------------
try:
    allowed_cores
except NameError:
    import psutil
    physical_cores = psutil.cpu_count(logical=False) or psutil.cpu_count(logical=True) or 1
    allowed_cores = max(1, int(physical_cores * 0.8))

try:
    cooling_pause
except NameError:
    def cooling_pause(stage="", force_short_pause=False):
        if force_short_pause:
            import time
            time.sleep(5)

print(f"Allowed CPU cores used inside estimators: {allowed_cores}")
print("CV-level n_jobs is set to 1 to avoid nested parallel CPU overload.")


# -----------------------------------------------------------------------------------------
# 1) 10-Fold CV scores for TRUE AOA optimized final pipelines
# Important:
# n_jobs=1 here is intentional.
# The RF/BRF estimators already use n_jobs=allowed_cores.
# -----------------------------------------------------------------------------------------
print("\n>>> Computing 10-Fold CV ROC-AUC scores for TRUE AOA Optimized Models...")

cv_outer = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

aoa_pipelines = {
    "AOA-BRF": pipe_brf_fs_calib,
    "AOA-RF+SMOTEENN": pipe_rf_sm_fs_calib,
    "AOA-BRF+SMOTEENN": pipe_brf_sm_fs_calib
}

all_folds = {}

for name, pipe in aoa_pipelines.items():
    cooling_pause(
        stage=f"before 10-fold CV for {name}",
        force_short_pause=True
    )

    print(f"Computing CV scores for {name}...")

    scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=cv_outer,
        scoring="roc_auc",
        n_jobs=1
    )

    all_folds[name] = scores

    print(
        f"{name:<25} | "
        f"Mean ROC-AUC={np.mean(scores):.5f} | "
        f"Std={np.std(scores):.5f}"
    )

    cooling_pause(
        stage=f"after 10-fold CV for {name}",
        force_short_pause=True
    )


# -----------------------------------------------------------------------------------------
# 2) Summary table
# -----------------------------------------------------------------------------------------
summary_df = pd.DataFrame([
    {
        "Model": name,
        "Mean_ROC_AUC": np.mean(scores),
        "Std_ROC_AUC": np.std(scores),
        "Min_ROC_AUC": np.min(scores),
        "Max_ROC_AUC": np.max(scores)
    }
    for name, scores in all_folds.items()
]).sort_values(
    by="Mean_ROC_AUC",
    ascending=False
).reset_index(drop=True)

print("\n--- 10-Fold ROC-AUC Summary ---")
print(summary_df)


# -----------------------------------------------------------------------------------------
# 3) Automatically select the best model based on mean ROC-AUC
# This is better than hard-coding AOA-BRF as the best model.
# -----------------------------------------------------------------------------------------
target_model = summary_df.iloc[0]["Model"]

print("\n" + "-" * 95)
print(f"Best model based on mean ROC-AUC: {target_model}")
print("-" * 95)


# -----------------------------------------------------------------------------------------
# 4) Paired statistical tests vs best model
# Paired t-test + Wilcoxon signed-rank test
# -----------------------------------------------------------------------------------------
print(
    f"{'Comparison vs ' + target_model:<45} | "
    f"{'T-Stat':<10} | {'T-P':<10} | "
    f"{'W-Stat':<10} | {'W-P':<10} | "
    f"{'Sig(T)':<8} | {'Sig(W)'}"
)
print("-" * 95)

for name, fold_data in all_folds.items():
    if name == target_model:
        continue

    t_stat, t_p = stats.ttest_rel(
        all_folds[target_model],
        fold_data
    )

    try:
        w_stat, w_p = stats.wilcoxon(
            all_folds[target_model],
            fold_data,
            zero_method="wilcox",
            alternative="two-sided"
        )
    except ValueError:
        w_stat, w_p = np.nan, np.nan

    sig_t = "YES" if t_p < 0.05 else "NO"
    sig_w = "YES" if not np.isnan(w_p) and w_p < 0.05 else "NO"

    print(
        f"{target_model + ' vs ' + name:<45} | "
        f"{t_stat:>10.4f} | {t_p:>10.4f} | "
        f"{w_stat:>10.4f} | {w_p:>10.4f} | "
        f"{sig_t:<8} | {sig_w}"
    )


# -----------------------------------------------------------------------------------------
# 5) Full pairwise comparison between all AOA models
# -----------------------------------------------------------------------------------------
print("\n--- Full Pairwise Paired T-Test Analysis ---")
print(f"{'Model Pair':<55} | {'T-Stat':<10} | {'P-Value':<10} | {'Significant'}")
print("-" * 95)

for name1, name2 in itertools.combinations(all_folds.keys(), 2):
    t_stat, p_val = stats.ttest_rel(
        all_folds[name1],
        all_folds[name2]
    )

    is_sig = "✅ YES" if p_val < 0.05 else "⚠️ NO"

    print(
        f"{name1 + ' vs ' + name2:<55} | "
        f"{t_stat:>10.4f} | "
        f"{p_val:>10.4f} | "
        f"{is_sig}"
    )


# -----------------------------------------------------------------------------------------
# 6) Stability report
# -----------------------------------------------------------------------------------------
print("\n" + "-" * 95)
print(f"{'AOA Model':<30} | {'Mean ROC-AUC':<12} | {'Std Dev':<10} | {'CV Scores'}")
print("-" * 95)

for name, fold_data in all_folds.items():
    print(
        f"{name:<30} | "
        f"{np.mean(fold_data):.5f}      | "
        f"{np.std(fold_data):.5f}   | "
        f"{np.round(fold_data, 4)}"
    )

print("=" * 95 + "\n")


# -----------------------------------------------------------------------------------------
# 7) Boxplot for statistical stability
# -----------------------------------------------------------------------------------------
plt.figure(figsize=(10, 6))

try:
    box = plt.boxplot(
        all_folds.values(),
        patch_artist=True,
        tick_labels=list(all_folds.keys()),
        widths=0.5
    )
except TypeError:
    box = plt.boxplot(
        all_folds.values(),
        patch_artist=True,
        labels=list(all_folds.keys()),
        widths=0.5
    )

colors = ["#1D3557", "#E63946", "#457B9D"]

for patch, color in zip(box["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

plt.title(
    "Statistical Stability: TRUE AOA Optimized Pipelines",
    fontsize=13,
    fontweight="bold"
)

plt.ylabel(
    "Validation ROC-AUC Score (10-Fold CV)",
    fontsize=11,
    fontweight="bold"
)

plt.grid(
    True,
    linestyle="--",
    alpha=0.3,
    axis="y"
)

plt.tight_layout()
plt.show()


# -----------------------------------------------------------------------------------------
# 8) Save results for paper tables
# -----------------------------------------------------------------------------------------
summary_df.to_csv(
    "aoa_statistical_summary.csv",
    index=False
)

folds_df = pd.DataFrame(all_folds)
folds_df.to_csv(
    "aoa_10fold_roc_auc_scores.csv",
    index=False
)

print("✅ Statistical analysis for TRUE AOA models completed.")
print("• Saved files: aoa_statistical_summary.csv, aoa_10fold_roc_auc_scores.csv")


# -----------------------------------------------------------------------------------------
# 9) Save all currently open output figures to local system
# -----------------------------------------------------------------------------------------
import os
from datetime import datetime

fig_output_dir = "saved_figures_aoa_statistical_analysis"
os.makedirs(fig_output_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

open_figures = plt.get_fignums()

if len(open_figures) == 0:
    print("⚠️ No open matplotlib figures found to save.")
    print("Note: If your backend closes figures after plt.show(), savefig should be placed before plt.show().")
else:
    saved_paths = []

    for i, fig_num in enumerate(open_figures, start=1):
        fig = plt.figure(fig_num)

        png_path = os.path.join(
            fig_output_dir,
            f"aoa_statistical_analysis_figure_{i}_{timestamp}.png"
        )

        pdf_path = os.path.join(
            fig_output_dir,
            f"aoa_statistical_analysis_figure_{i}_{timestamp}.pdf"
        )

        tif_path = os.path.join(
            fig_output_dir,
            f"aoa_statistical_analysis_figure_{i}_{timestamp}.tif"
        )

        fig.savefig(png_path, dpi=300, bbox_inches="tight")
        fig.savefig(pdf_path, bbox_inches="tight")
        fig.savefig(tif_path, dpi=300, bbox_inches="tight")

        saved_paths.extend([png_path, pdf_path, tif_path])

    print("✅ All currently open figures were saved successfully.")
    print("Saved figure files:")
    for path in saved_paths:
        print("•", path)

In [ ]:
# ===== Block 18: Full Pipelines — PI-based FS + TRUE AOA++ Optimization + Calibration + Evaluation =====
# ===== Train/Test Split, Approximate CPU Control, Cooling Pauses, and Evaluation Output =====

import os
import random
import time
import psutil

# -----------------------------------------------------------------------------------------
# CPU LIMIT: use about 80% of CPU capacity
# -----------------------------------------------------------------------------------------
physical_cores = psutil.cpu_count(logical=False) or psutil.cpu_count(logical=True) or 1
allowed_cores = max(1, int(physical_cores * 0.8))

os.environ["OMP_NUM_THREADS"] = str(allowed_cores)
os.environ["MKL_NUM_THREADS"] = str(allowed_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(allowed_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(allowed_cores)

try:
    proc = psutil.Process(os.getpid())
    available_cpus = proc.cpu_affinity()
    keep_n = max(1, int(len(available_cpus) * 0.8))
    proc.cpu_affinity(available_cpus[:keep_n])
except Exception:
    pass

COOL_EVERY_ITERS = 1
COOL_DOWN_SECONDS = 5

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.metrics import (
    roc_auc_score, recall_score, f1_score, accuracy_score,
    precision_score, confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_predict, cross_val_score
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier

from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.combine import SMOTEENN
from imblearn.ensemble import BalancedRandomForestClassifier


def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_all_seeds(42)


# -----------------------------------------------------------------------------------------
# 0) LEAK-SAFE TRAIN/TEST SPLIT
# Dataset-specific variables from Code 1:
# X, y, num_cols, cat_cols must already exist.
# -----------------------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


# -----------------------------------------------------------------------------------------
# 1) Permutation Importance → Top-K
# Same dataset-specific feature selection style as Code 1
# -----------------------------------------------------------------------------------------
K = 15
top_features, imp_df = compute_perm_importance_topk(
    X_train, y_train, num_cols, cat_cols, K=K, seed=42
)

print(f"Selected Top-{K} features:", top_features)


# -----------------------------------------------------------------------------------------
# 2) Preprocessing pipeline
# -----------------------------------------------------------------------------------------
num_pipe = SkPipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

cat_pipe = SkPipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

base_pre = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])


# -----------------------------------------------------------------------------------------
# FeatureNameSelector fallback
# If you already defined FeatureNameSelector before, this will not overwrite it.
# -----------------------------------------------------------------------------------------
try:
    FeatureNameSelector
except NameError:

    class FeatureNameSelector(BaseEstimator, TransformerMixin):
        def __init__(self, preprocessor, selected_feature_names):
            self.preprocessor = preprocessor
            self.selected_feature_names = selected_feature_names

        def fit(self, X, y=None):
            self.preprocessor_ = clone(self.preprocessor)
            self.preprocessor_.fit(X, y)

            try:
                feature_names = self.preprocessor_.get_feature_names_out()
            except Exception:
                Xt = self.preprocessor_.transform(X)
                feature_names = np.array([f"f{i}" for i in range(Xt.shape[1])])

            self.feature_names_out_ = np.array(feature_names)
            selected = set(map(str, self.selected_feature_names))

            selected_indices = []

            for idx, fname in enumerate(self.feature_names_out_):
                clean_name = str(fname).split("__", 1)[-1]

                keep = False

                if clean_name in selected:
                    keep = True

                for s in selected:
                    if clean_name == s or clean_name.startswith(s + "_"):
                        keep = True
                        break

                if keep:
                    selected_indices.append(idx)

            if len(selected_indices) == 0:
                raise ValueError(
                    "No selected features matched transformed feature names. "
                    "Check top_features, num_cols, cat_cols, and FeatureNameSelector."
                )

            self.selected_indices_ = np.array(selected_indices)
            return self

        def transform(self, X):
            Xt = self.preprocessor_.transform(X)
            return Xt[:, self.selected_indices_]


def make_calibrated_classifier(base_estimator):
    try:
        return CalibratedClassifierCV(
            estimator=base_estimator,
            method="isotonic",
            cv=5
        )
    except TypeError:
        return CalibratedClassifierCV(
            base_estimator=base_estimator,
            method="isotonic",
            cv=5
        )


def make_fs_calibrated_pipeline(base_estimator, selected_feature_names, sampler=None):
    feat_sel = FeatureNameSelector(
        preprocessor=base_pre,
        selected_feature_names=selected_feature_names
    )

    model_cal = make_calibrated_classifier(base_estimator)

    steps = [("feat_pre", feat_sel)]

    if sampler is not None:
        steps.append(("sampler", sampler))

    steps.append(("model", model_cal))

    return ImbPipeline(steps)


# -----------------------------------------------------------------------------------------
# 3) TRUE AOA++ SEARCH SPACE
# We optimize normalized numeric vectors Xi in [0,1]^d.
# Then decode each vector to actual RF / BRF hyperparameters.
# -----------------------------------------------------------------------------------------
MAX_DEPTH_CHOICES = [None, 3, 5, 7, 10]
MAX_FEATURES_CHOICES = ["sqrt", "log2", None]


def _decode_int(z, low, high):
    z = float(np.clip(z, 0, 1))
    return int(round(low + z * (high - low)))


def _decode_choice(z, choices):
    z = float(np.clip(z, 0, 1))
    idx = int(round(z * (len(choices) - 1)))
    idx = int(np.clip(idx, 0, len(choices) - 1))
    return choices[idx]


def decode_candidate_vector(x_vec):
    """
    x_vec = [n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features]
    all values are normalized in [0,1].
    """
    x_vec = np.clip(np.asarray(x_vec, dtype=float), 0, 1)

    return {
        "n_estimators": _decode_int(x_vec[0], 150, 600),
        "max_depth": _decode_choice(x_vec[1], MAX_DEPTH_CHOICES),
        "min_samples_split": _decode_int(x_vec[2], 2, 12),
        "min_samples_leaf": _decode_int(x_vec[3], 1, 6),
        "max_features": _decode_choice(x_vec[4], MAX_FEATURES_CHOICES),
    }


# -----------------------------------------------------------------------------------------
# 3.1) Estimators
# CPU-limited with n_jobs=allowed_cores
# -----------------------------------------------------------------------------------------
def build_brf(p):
    return BalancedRandomForestClassifier(
        n_estimators=p["n_estimators"],
        max_depth=p["max_depth"],
        min_samples_split=p["min_samples_split"],
        min_samples_leaf=p["min_samples_leaf"],
        max_features=p["max_features"],
        random_state=42,
        n_jobs=allowed_cores
    )


def build_rf(p):
    return RandomForestClassifier(
        n_estimators=p["n_estimators"],
        max_depth=p["max_depth"],
        min_samples_split=p["min_samples_split"],
        min_samples_leaf=p["min_samples_leaf"],
        max_features=p["max_features"],
        random_state=42,
        n_jobs=allowed_cores
    )


# -----------------------------------------------------------------------------------------
# 4) TRUE AOA++ OPTIMIZER
# Algorithm:
# Initialize Xi
# alpha_max = 2.0, gamma = 0.1
# Xbest = initial best solution with lowest fitness
# for t = 1 to T:
#     XM = mean(population)
#     alpha = alpha_max * exp(-gamma * t / T)
#     X'i = Xbest + alpha * (rand - 0.5) * (XM - Xi)
#     evaluate fitness
#     if fitness(X'i) < fitness(Xbest): Xbest = X'i
# return Xbest
# -----------------------------------------------------------------------------------------
def aoa_optimize_generic(
    build_estimator_fn,
    selected_feature_names,
    use_smoteen=False,
    max_iter=15,
    pop_size=24,
    seed=42,
    label="GEN",
    alpha_max=2.0,
    gamma=0.1
):
    start_opt_time = time.time()

    rng = np.random.RandomState(seed)
    sampler = SMOTEENN(random_state=seed) if use_smoteen else None

    # Internal validation set for AOA++ fitness evaluation
    X_aoa_train, X_aoa_val, y_aoa_train, y_aoa_val = train_test_split(
        X_train,
        y_train,
        test_size=0.2,
        stratify=y_train,
        random_state=seed
    )

    def fitness(x_vec):
        """
        Algorithm minimizes fitness.
        Since we want high AUC/Recall/F1/Accuracy, we define:
        fitness = -weighted_score
        """
        params = decode_candidate_vector(x_vec)

        est = build_estimator_fn(params)

        pipe = make_fs_calibrated_pipeline(
            est,
            selected_feature_names,
            sampler=sampler
        )

        pipe.fit(X_aoa_train, y_aoa_train)

        val_proba = pipe.predict_proba(X_aoa_val)[:, 1]
        val_pred = (val_proba >= 0.5).astype(int)

        auc = roc_auc_score(y_aoa_val, val_proba)
        rec = recall_score(y_aoa_val, val_pred, zero_division=0)
        f1 = f1_score(y_aoa_val, val_pred, zero_division=0)
        acc = accuracy_score(y_aoa_val, val_pred)

        score = 0.4 * auc + 0.4 * rec + 0.1 * f1 + 0.1 * acc

        fit_value = -score

        return fit_value, {
            "auc": auc,
            "recall": rec,
            "f1": f1,
            "acc": acc,
            "score": score,
            **params
        }

    # Initialize population Xi
    dim = 5
    population = rng.uniform(0, 1, size=(pop_size, dim))

    # Initial best solution with lowest fitness
    best_x = None
    best_fit = np.inf
    best_mets = None
    history = []

    for i in range(pop_size):
        fit_i, mets_i = fitness(population[i])

        history.append({
            "iter": 0,
            "individual": i + 1,
            "alpha": np.nan,
            "fitness": fit_i,
            **mets_i
        })

        if fit_i < best_fit:
            best_x = population[i].copy()
            best_fit = fit_i
            best_mets = mets_i.copy()

    # AOA++ main loop
    for t in range(1, max_iter + 1):

        # XM ← mean of all Xi in population
        XM = np.mean(population, axis=0)

        # alpha ← alpha_max * exp(-gamma * t / T)
        alpha = alpha_max * np.exp(-gamma * t / max_iter)

        new_population = []

        for i in range(pop_size):
            Xi = population[i].copy()

            # rand ← uniform random number in [0, 1]
            rand = rng.uniform(0, 1)

            # X'i ← Xbest + alpha * (rand - 0.5) * (XM - Xi)
            Xi_prime = best_x + alpha * (rand - 0.5) * (XM - Xi)

            # Keep normalized vector inside valid bounds
            Xi_prime = np.clip(Xi_prime, 0, 1)

            # Evaluate fitness f(X'i)
            fit_prime, mets_prime = fitness(Xi_prime)

            history.append({
                "iter": t,
                "individual": i + 1,
                "alpha": alpha,
                "fitness": fit_prime,
                **mets_prime
            })

            # if f(X'i) < f(Xbest), update Xbest
            if fit_prime < best_fit:
                best_x = Xi_prime.copy()
                best_fit = fit_prime
                best_mets = mets_prime.copy()

            # Update position
            new_population.append(Xi_prime)

        population = np.array(new_population)

        print(
            f"[{label}] Iter {t}/{max_iter} | "
            f"Score={best_mets['score']:.4f} | "
            f"AUC={best_mets['auc']:.4f} | "
            f"F1={best_mets['f1']:.4f} | "
            f"ACC={best_mets['acc']:.4f} | "
            f"Recall={best_mets['recall']:.4f}"
        )

        if COOL_DOWN_SECONDS > 0 and t % COOL_EVERY_ITERS == 0:
            time.sleep(COOL_DOWN_SECONDS)

    best_params = decode_candidate_vector(best_x)

    # Save fold scores for statistical testing, like Code 2
    best_pipe = make_fs_calibrated_pipeline(
        build_estimator_fn(best_params),
        selected_feature_names,
        sampler=sampler
    )

    cv_inner = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=seed
    )

    best_folds = cross_val_score(
        best_pipe,
        X_train,
        y_train,
        cv=cv_inner,
        scoring="roc_auc",
        n_jobs=1
    )

    duration = time.time() - start_opt_time
    print(f"⏱️ Optimization for [{label}] completed in {duration:.2f} seconds.\n")

    return best_params, best_mets, best_folds, pd.DataFrame(history)


# -----------------------------------------------------------------------------------------
# 5) Run TRUE AOA++
# -----------------------------------------------------------------------------------------
print(">>> Running TRUE AOA++ Optimization...")

best_params_brf, best_mets_brf, folds_brf, hist_brf = aoa_optimize_generic(
    build_brf,
    top_features,
    use_smoteen=False,
    seed=42,
    label="BRF"
)

best_params_rf_sm, best_mets_rf_sm, folds_rf_sm, hist_rf_sm = aoa_optimize_generic(
    build_rf,
    top_features,
    use_smoteen=True,
    seed=42,
    label="RF+SMOTEENN"
)

best_params_brf_sm, best_mets_brf_sm, folds_brf_sm, hist_brf_sm = aoa_optimize_generic(
    build_brf,
    top_features,
    use_smoteen=True,
    seed=42,
    label="BRF+SMOTEENN"
)


# -----------------------------------------------------------------------------------------
# 6) FINAL PIPELINES
# -----------------------------------------------------------------------------------------
pipe_brf_fs_calib = make_fs_calibrated_pipeline(
    build_brf(best_params_brf),
    top_features
)

pipe_rf_sm_fs_calib = make_fs_calibrated_pipeline(
    build_rf(best_params_rf_sm),
    top_features,
    sampler=SMOTEENN(random_state=42)
)

pipe_brf_sm_fs_calib = make_fs_calibrated_pipeline(
    build_brf(best_params_brf_sm),
    top_features,
    sampler=SMOTEENN(random_state=42)
)


# -----------------------------------------------------------------------------------------
# 7) CV object for OOF
# -----------------------------------------------------------------------------------------
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)


# -----------------------------------------------------------------------------------------
# 8) THRESHOLD OPTIMIZATION FUNCTION
# -----------------------------------------------------------------------------------------
def find_best_threshold(y_true, y_proba, metric="f1"):
    thresholds = np.linspace(0, 1, 200)
    best_thr = 0.5
    best_score = -1

    for thr in thresholds:
        preds = (y_proba >= thr).astype(int)

        if metric == "f1":
            score = f1_score(y_true, preds, zero_division=0)
        elif metric == "recall":
            score = recall_score(y_true, preds, zero_division=0)
        elif metric == "precision":
            score = precision_score(y_true, preds, zero_division=0)
        else:
            score = f1_score(y_true, preds, zero_division=0)

        if score > best_score:
            best_score = score
            best_thr = thr

    return best_thr, best_score


# -----------------------------------------------------------------------------------------
# 9) EVALUATION BLOCKS
# Same output style as Code 2
# -----------------------------------------------------------------------------------------
def oof_block(pipe, X_tr, y_tr):
    proba_tr = cross_val_predict(
        pipe,
        X_tr,
        y_tr,
        cv=cv,
        method="predict_proba",
        n_jobs=1
    )[:, 1]

    pred_tr_05 = (proba_tr >= 0.5).astype(int)

    t_opt, _ = find_best_threshold(
        y_tr,
        proba_tr,
        metric="f1"
    )

    pred_tr_opt = (proba_tr >= t_opt).astype(int)

    mets_tr_05 = {
        "accuracy": accuracy_score(y_tr, pred_tr_05),
        "roc_auc": roc_auc_score(y_tr, proba_tr),
        "recall": recall_score(y_tr, pred_tr_05, zero_division=0),
        "precision": precision_score(y_tr, pred_tr_05, zero_division=0),
        "f1": f1_score(y_tr, pred_tr_05, zero_division=0),
    }

    mets_tr_opt = {
        "accuracy": accuracy_score(y_tr, pred_tr_opt),
        "roc_auc": roc_auc_score(y_tr, proba_tr),
        "recall": recall_score(y_tr, pred_tr_opt, zero_division=0),
        "precision": precision_score(y_tr, pred_tr_opt, zero_division=0),
        "f1": f1_score(y_tr, pred_tr_opt, zero_division=0),
        "threshold": t_opt,
    }

    return proba_tr, mets_tr_05, mets_tr_opt, t_opt


def test_block(pipe, X_te, y_te, t_opt):
    proba_te = pipe.predict_proba(X_te)[:, 1]

    pred_te_05 = (proba_te >= 0.5).astype(int)
    pred_te_opt = (proba_te >= t_opt).astype(int)

    mets_te_05 = {
        "accuracy": accuracy_score(y_te, pred_te_05),
        "roc_auc": roc_auc_score(y_te, proba_te),
        "recall": recall_score(y_te, pred_te_05, zero_division=0),
        "precision": precision_score(y_te, pred_te_05, zero_division=0),
        "f1": f1_score(y_te, pred_te_05, zero_division=0),
    }

    mets_te_opt = {
        "accuracy": accuracy_score(y_te, pred_te_opt),
        "roc_auc": roc_auc_score(y_te, proba_te),
        "recall": recall_score(y_te, pred_te_opt, zero_division=0),
        "precision": precision_score(y_te, pred_te_opt, zero_division=0),
        "f1": f1_score(y_te, pred_te_opt, zero_division=0),
        "threshold": t_opt,
    }

    return proba_te, mets_te_05, mets_te_opt


def full_eval(pipe, name):
    proba_tr, tr05, tropt, t_opt = oof_block(
        pipe,
        X_train,
        y_train
    )

    pipe.fit(X_train, y_train)

    if COOL_DOWN_SECONDS > 0:
        time.sleep(COOL_DOWN_SECONDS)

    proba_te, te05, teopt = test_block(
        pipe,
        X_test,
        y_test,
        t_opt
    )

    print(f"\n=== {name} Calibrated with PI-TopK ===")
    print("Train (OOF) @0.50:", {k: round(v, 4) for k, v in tr05.items()})
    print("Train (OOF) @opt :", {k: round(v, 4) for k, v in tropt.items()})
    print("Test        @0.50:", {k: round(v, 4) for k, v in te05.items()})
    print("Test        @opt :", {k: round(v, 4) for k, v in teopt.items()})

    # ROC & PR plots - Train
    plt.figure(figsize=(14, 6))

    plt.subplot(1, 2, 1)
    RocCurveDisplay.from_predictions(
        y_train,
        proba_tr,
        ax=plt.gca()
    )
    plt.title(f"ROC - Train (OOF) [{name}]")

    plt.subplot(1, 2, 2)
    PrecisionRecallDisplay.from_predictions(
        y_train,
        proba_tr,
        ax=plt.gca()
    )
    plt.title(f"PR - Train (OOF) [{name}]")

    plt.tight_layout()
    plt.show()

    # ROC & PR plots - Test
    plt.figure(figsize=(14, 6))

    plt.subplot(1, 2, 1)
    RocCurveDisplay.from_predictions(
        y_test,
        proba_te,
        ax=plt.gca()
    )
    plt.title(f"ROC - Test [{name}]")

    plt.subplot(1, 2, 2)
    PrecisionRecallDisplay.from_predictions(
        y_test,
        proba_te,
        ax=plt.gca()
    )
    plt.title(f"PR - Test [{name}]")

    plt.tight_layout()
    plt.show()

    # Confusion @0.50
    pred_tr_05 = (proba_tr >= 0.5).astype(int)
    pred_te_05 = (proba_te >= 0.5).astype(int)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    sns.heatmap(
        confusion_matrix(y_train, pred_tr_05),
        annot=True,
        fmt="d",
        cmap="Blues"
    )
    plt.title(f"Confusion Matrix - Train (OOF @0.5) [{name}]")

    plt.subplot(1, 2, 2)
    sns.heatmap(
        confusion_matrix(y_test, pred_te_05),
        annot=True,
        fmt="d",
        cmap="Blues"
    )
    plt.title(f"Confusion Matrix - Test (@0.5) [{name}]")

    plt.tight_layout()
    plt.show()

    # Confusion @OPT
    pred_tr_opt = (proba_tr >= t_opt).astype(int)
    pred_te_opt = (proba_te >= t_opt).astype(int)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    sns.heatmap(
        confusion_matrix(y_train, pred_tr_opt),
        annot=True,
        fmt="d",
        cmap="Greens"
    )
    plt.title(f"Train Confusion @opt={t_opt:.2f} [{name}]")

    plt.subplot(1, 2, 2)
    sns.heatmap(
        confusion_matrix(y_test, pred_te_opt),
        annot=True,
        fmt="d",
        cmap="Greens"
    )
    plt.title(f"Test Confusion @opt={t_opt:.2f} [{name}]")

    plt.tight_layout()
    plt.show()

    return dict(
        train_05=tr05,
        train_opt=tropt,
        test_05=te05,
        test_opt=teopt
    )


# -----------------------------------------------------------------------------------------
# 10) FINAL RESULTS EXECUTION
# -----------------------------------------------------------------------------------------
res_brf = full_eval(
    pipe_brf_fs_calib,
    "BRF"
)

if COOL_DOWN_SECONDS > 0:
    time.sleep(COOL_DOWN_SECONDS)

res_rf_sm = full_eval(
    pipe_rf_sm_fs_calib,
    "RF+SMOTEENN"
)

if COOL_DOWN_SECONDS > 0:
    time.sleep(COOL_DOWN_SECONDS)

res_brf_sm = full_eval(
    pipe_brf_sm_fs_calib,
    "BRF+SMOTEENN"
)

In [ ]:
# ===== Block 19: Statistical Analysis for TRUE AOA++ Optimized Models =====
# ===== CPU-80% Compatible + Temperature-Aware Cooling =====
# This block is written in the same structure as Block 21 for TRUE AOA models.

import itertools
import scipy.stats as stats
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from datetime import datetime

print("=" * 95)
print(" STEP 4: Statistical Comparison of TRUE AOA++ Optimized Models")
print("=" * 95)

# -----------------------------------------------------------------------------------------
# 0) CPU / cooling compatibility
# -----------------------------------------------------------------------------------------
try:
    allowed_cores
except NameError:
    import psutil
    physical_cores = psutil.cpu_count(logical=False) or psutil.cpu_count(logical=True) or 1
    allowed_cores = max(1, int(physical_cores * 0.8))

try:
    cooling_pause
except NameError:
    def cooling_pause(stage="", force_short_pause=False):
        if force_short_pause:
            import time
            time.sleep(5)

print(f"Allowed CPU cores used inside estimators: {allowed_cores}")
print("CV-level n_jobs is set to 1 to avoid nested parallel CPU overload.")


# -----------------------------------------------------------------------------------------
# 1) Use saved 10-fold ROC-AUC scores from TRUE AOA++ optimization
# Required variables from previous AOA++ block:
# folds_brf, folds_rf_sm, folds_brf_sm
# -----------------------------------------------------------------------------------------
required_fold_variables = ["folds_brf", "folds_rf_sm", "folds_brf_sm"]

missing_variables = [
    var_name for var_name in required_fold_variables
    if var_name not in globals()
]

if len(missing_variables) > 0:
    raise NameError(
        "The following AOA++ fold-score variables are missing: "
        + ", ".join(missing_variables)
        + "\nRun the TRUE AOA++ optimization block before this statistical analysis block."
    )

print("\n>>> Loading 10-Fold CV ROC-AUC scores for TRUE AOA++ Optimized Models...")

aoapp_all_folds = {
    "AOA++-BRF": np.asarray(folds_brf, dtype=float),
    "AOA++-RF+SMOTEENN": np.asarray(folds_rf_sm, dtype=float),
    "AOA++-BRF+SMOTEENN": np.asarray(folds_brf_sm, dtype=float)
}

fold_lengths = [len(scores) for scores in aoapp_all_folds.values()]

if len(set(fold_lengths)) != 1:
    raise ValueError(
        "The AOA++ fold-score arrays do not have the same length. "
        f"Detected lengths: {dict(zip(aoapp_all_folds.keys(), fold_lengths))}"
    )

for name, scores in aoapp_all_folds.items():
    print(
        f"{name:<30} | "
        f"Mean ROC-AUC={np.mean(scores):.5f} | "
        f"Std={np.std(scores):.5f}"
    )


# -----------------------------------------------------------------------------------------
# 2) Summary table
# -----------------------------------------------------------------------------------------
aoapp_summary_df = pd.DataFrame([
    {
        "Model": name,
        "Mean_ROC_AUC": np.mean(scores),
        "Std_ROC_AUC": np.std(scores),
        "Min_ROC_AUC": np.min(scores),
        "Max_ROC_AUC": np.max(scores)
    }
    for name, scores in aoapp_all_folds.items()
]).sort_values(
    by="Mean_ROC_AUC",
    ascending=False
).reset_index(drop=True)

print("\n--- 10-Fold ROC-AUC Summary ---")
print(aoapp_summary_df)


# -----------------------------------------------------------------------------------------
# 3) Automatically select the best model based on mean ROC-AUC
# -----------------------------------------------------------------------------------------
target_model = aoapp_summary_df.iloc[0]["Model"]

print("\n" + "-" * 95)
print(f"Best model based on mean ROC-AUC: {target_model}")
print("-" * 95)


# -----------------------------------------------------------------------------------------
# 4) Paired statistical tests vs best model
# Paired t-test + Wilcoxon signed-rank test
# -----------------------------------------------------------------------------------------
print(
    f"{'Comparison vs ' + target_model:<50} | "
    f"{'T-Stat':<10} | {'T-P':<10} | "
    f"{'W-Stat':<10} | {'W-P':<10} | "
    f"{'Sig(T)':<8} | {'Sig(W)'}"
)
print("-" * 95)

for name, fold_data in aoapp_all_folds.items():
    if name == target_model:
        continue

    t_stat, t_p = stats.ttest_rel(
        aoapp_all_folds[target_model],
        fold_data
    )

    try:
        w_stat, w_p = stats.wilcoxon(
            aoapp_all_folds[target_model],
            fold_data,
            zero_method="wilcox",
            alternative="two-sided"
        )
    except ValueError:
        w_stat, w_p = np.nan, np.nan

    sig_t = "YES" if t_p < 0.05 else "NO"
    sig_w = "YES" if not np.isnan(w_p) and w_p < 0.05 else "NO"

    print(
        f"{target_model + ' vs ' + name:<50} | "
        f"{t_stat:>10.4f} | {t_p:>10.4f} | "
        f"{w_stat:>10.4f} | {w_p:>10.4f} | "
        f"{sig_t:<8} | {sig_w}"
    )


# -----------------------------------------------------------------------------------------
# 5) Full pairwise comparison between all AOA++ models
# Paired t-test for all model pairs
# -----------------------------------------------------------------------------------------
print("\n--- Full Pairwise Paired T-Test Analysis ---")
print(f"{'Model Pair':<60} | {'T-Stat':<10} | {'P-Value':<10} | {'Significant'}")
print("-" * 95)

for name1, name2 in itertools.combinations(aoapp_all_folds.keys(), 2):
    t_stat, p_val = stats.ttest_rel(
        aoapp_all_folds[name1],
        aoapp_all_folds[name2]
    )

    is_sig = "✅ YES" if p_val < 0.05 else "⚠️ NO"

    print(
        f"{name1 + ' vs ' + name2:<60} | "
        f"{t_stat:>10.4f} | "
        f"{p_val:>10.4f} | "
        f"{is_sig}"
    )


# -----------------------------------------------------------------------------------------
# 6) Stability report
# -----------------------------------------------------------------------------------------
print("\n" + "-" * 95)
print(f"{'AOA++ Model':<35} | {'Mean ROC-AUC':<12} | {'Std Dev':<10} | {'CV Scores'}")
print("-" * 95)

for name, fold_data in aoapp_all_folds.items():
    print(
        f"{name:<35} | "
        f"{np.mean(fold_data):.5f}      | "
        f"{np.std(fold_data):.5f}   | "
        f"{np.round(fold_data, 4)}"
    )

print("=" * 95 + "\n")


# -----------------------------------------------------------------------------------------
# 7) Boxplot for statistical stability
# -----------------------------------------------------------------------------------------
plt.figure(figsize=(10, 6))

try:
    box = plt.boxplot(
        aoapp_all_folds.values(),
        patch_artist=True,
        tick_labels=list(aoapp_all_folds.keys()),
        widths=0.5
    )
except TypeError:
    box = plt.boxplot(
        aoapp_all_folds.values(),
        patch_artist=True,
        labels=list(aoapp_all_folds.keys()),
        widths=0.5
    )

colors = ["#1D3557", "#E63946", "#457B9D"]

for patch, color in zip(box["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

plt.title(
    "Statistical Stability: TRUE AOA++ Optimized Pipelines",
    fontsize=13,
    fontweight="bold"
)

plt.ylabel(
    "Validation ROC-AUC Score (10-Fold CV)",
    fontsize=11,
    fontweight="bold"
)

plt.grid(
    True,
    linestyle="--",
    alpha=0.3,
    axis="y"
)

plt.tight_layout()


# -----------------------------------------------------------------------------------------
# 8) Save results for paper tables
# -----------------------------------------------------------------------------------------
aoapp_summary_df.to_csv(
    "aoapp_statistical_summary.csv",
    index=False
)

aoapp_folds_df = pd.DataFrame(aoapp_all_folds)
aoapp_folds_df.to_csv(
    "aoapp_10fold_roc_auc_scores.csv",
    index=False
)

print("✅ Statistical analysis for TRUE AOA++ models completed.")
print("• Saved files: aoapp_statistical_summary.csv, aoapp_10fold_roc_auc_scores.csv")


# -----------------------------------------------------------------------------------------
# 9) Save all output figures before plt.show()
# -----------------------------------------------------------------------------------------
fig_output_dir = "saved_figures_aoapp_statistical_analysis"
os.makedirs(fig_output_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

open_figures = plt.get_fignums()

if len(open_figures) == 0:
    print("⚠️ No open matplotlib figures found to save.")
else:
    saved_paths = []

    for i, fig_num in enumerate(open_figures, start=1):
        fig = plt.figure(fig_num)

        png_path = os.path.join(
            fig_output_dir,
            f"aoapp_statistical_analysis_figure_{i}_{timestamp}.png"
        )

        pdf_path = os.path.join(
            fig_output_dir,
            f"aoapp_statistical_analysis_figure_{i}_{timestamp}.pdf"
        )

        tif_path = os.path.join(
            fig_output_dir,
            f"aoapp_statistical_analysis_figure_{i}_{timestamp}.tif"
        )

        fig.savefig(png_path, dpi=300, bbox_inches="tight")
        fig.savefig(pdf_path, bbox_inches="tight")
        fig.savefig(tif_path, dpi=300, bbox_inches="tight")

        saved_paths.extend([png_path, pdf_path, tif_path])

    print("✅ All currently open figures were saved successfully.")
    print("Saved figure files:")
    for path in saved_paths:
        print("•", path)

plt.show()

In [ ]:
# ===== Block 20: SHAP Analysis and Feature Importance Visualization =====


import os
import time
import psutil
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from datetime import datetime

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold

print("\n>>> Starting Fold-Stability-Validated SHAP Analysis for TRUE AOA++ Models...")

# -----------------------------------------------------------------------------------------
# 0) CPU / cooling compatibility
# -----------------------------------------------------------------------------------------
try:
    allowed_cores
except NameError:
    physical_cores = psutil.cpu_count(logical=False) or psutil.cpu_count(logical=True) or 1
    allowed_cores = max(1, int(physical_cores * 0.8))

os.environ["OMP_NUM_THREADS"] = str(allowed_cores)
os.environ["MKL_NUM_THREADS"] = str(allowed_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(allowed_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(allowed_cores)

try:
    COOL_DOWN_SECONDS
except NameError:
    COOL_DOWN_SECONDS = 5

RANDOM_STATE_SHAP = 42
SHAP_STABILITY_N_SPLITS = 5
SHAP_STABILITY_TOP_N = 10

SHAP_MAX_SAMPLES_PER_FOLD = None


# -----------------------------------------------------------------------------------------
# Extra) Figure output directory
# -----------------------------------------------------------------------------------------
fig_output_dir = "saved_figures_shap_analysis"
os.makedirs(fig_output_dir, exist_ok=True)

fig_save_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")


def save_current_figure_all_formats(base_name):
    fig = plt.gcf()

    png_path = os.path.join(
        fig_output_dir,
        f"{base_name}_{fig_save_timestamp}.png"
    )

    pdf_path = os.path.join(
        fig_output_dir,
        f"{base_name}_{fig_save_timestamp}.pdf"
    )

    tif_path = os.path.join(
        fig_output_dir,
        f"{base_name}_{fig_save_timestamp}.tif"
    )

    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(tif_path, dpi=300, bbox_inches="tight")

    print("Additional saved figure files:")
    print("•", png_path)
    print("•", pdf_path)
    print("•", tif_path)



final_pipe = pipe_brf_fs_calib
final_model_name = "TRUE AOA++ BRF"


# -----------------------------------------------------------------------------------------
# 2) Utility functions
# -----------------------------------------------------------------------------------------
def safe_index_rows(X, idx):
    if hasattr(X, "iloc"):
        return X.iloc[idx].copy()
    return np.asarray(X)[idx]


def safe_index_vector(y, idx):
    if hasattr(y, "iloc"):
        return y.iloc[idx].copy()
    return np.asarray(y)[idx]


def to_dense_array(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)


def make_unique_feature_names(names):
    counts = {}
    unique_names = []

    for name in names:
        name = str(name)

        if name not in counts:
            counts[name] = 0
            unique_names.append(name)
        else:
            counts[name] += 1
            unique_names.append(f"{name}_{counts[name]}")

    return unique_names


def is_identifier_feature_name(name):
    clean_name = str(name).split("__", 1)[-1].strip().lower()

    base_name = clean_name
    parts = clean_name.rsplit("_", 1)
    if len(parts) == 2 and parts[1].isdigit():
        base_name = parts[0]

    identifier_names = {
        "id",
        "patient_id",
        "patientid",
        "record_id",
        "recordid",
        "index",
        "unnamed: 0"
    }

    return clean_name in identifier_names or base_name in identifier_names


def remove_identifier_columns_from_input(X):
    if hasattr(X, "columns"):
        cols_to_drop = [
            c for c in X.columns
            if is_identifier_feature_name(c)
        ]

        if len(cols_to_drop) > 0:
            return X.drop(columns=cols_to_drop).copy()

    return X


def remove_identifier_columns_from_column_spec(columns):
    if isinstance(columns, (list, tuple, np.ndarray, pd.Index)):
        return [
            c for c in list(columns)
            if not is_identifier_feature_name(c)
        ]

    return columns


def remove_identifier_features_from_pipeline(pipe):
    if "feat_pre" not in pipe.named_steps:
        return pipe

    feat_step = pipe.named_steps["feat_pre"]

    if hasattr(feat_step, "selected_feature_names"):
        feat_step.selected_feature_names = [
            f for f in feat_step.selected_feature_names
            if not is_identifier_feature_name(f)
        ]

    if hasattr(feat_step, "preprocessor") and hasattr(feat_step.preprocessor, "transformers"):
        new_transformers = []

        for name, transformer, columns in feat_step.preprocessor.transformers:
            new_columns = remove_identifier_columns_from_column_spec(columns)
            new_transformers.append((name, transformer, new_columns))

        feat_step.preprocessor.transformers = new_transformers

    return pipe


def get_processed_feature_names(feat_selector, X_processed):
    feature_names = None

    if hasattr(feat_selector, "feature_names_out_") and hasattr(feat_selector, "selected_indices_"):
        all_names = np.asarray(feat_selector.feature_names_out_)
        selected_idx = np.asarray(feat_selector.selected_indices_)
        feature_names = all_names[selected_idx]

    elif hasattr(feat_selector, "get_feature_names_out"):
        try:
            feature_names = feat_selector.get_feature_names_out()
        except Exception:
            feature_names = None

    if feature_names is None or len(feature_names) != X_processed.shape[1]:
        feature_names = [f"feature_{i}" for i in range(X_processed.shape[1])]
    else:
        feature_names = [
            str(f).split("__", 1)[-1]
            for f in feature_names
        ]

    return make_unique_feature_names(feature_names)


def extract_base_estimator(calibrated_clf):
    if hasattr(calibrated_clf, "estimator"):
        return calibrated_clf.estimator

    if hasattr(calibrated_clf, "base_estimator"):
        return calibrated_clf.base_estimator

    if hasattr(calibrated_clf, "classifier"):
        return calibrated_clf.classifier

    raise AttributeError(
        "Could not extract base estimator from calibrated classifier. "
        "Check your sklearn version."
    )


def get_class1_shap_values(raw_shap_values):
    if isinstance(raw_shap_values, list):
        return raw_shap_values[1]

    raw_shap_values = np.asarray(raw_shap_values)

    if raw_shap_values.ndim == 3:
        # Common newer SHAP format: samples × features × classes
        if raw_shap_values.shape[2] == 2:
            return raw_shap_values[:, :, 1]

        # Alternative format: classes × samples × features
        if raw_shap_values.shape[0] == 2:
            return raw_shap_values[1, :, :]

    if raw_shap_values.ndim == 2:
        return raw_shap_values

    raise ValueError(
        f"Unsupported SHAP values shape: {raw_shap_values.shape}"
    )


def compute_shap_for_fitted_pipeline(fitted_pipe, X_input, context_name=""):
    feat_step = fitted_pipe.named_steps["feat_pre"]

    X_processed = feat_step.transform(X_input)
    X_processed = to_dense_array(X_processed)

    feature_names = get_processed_feature_names(feat_step, X_processed)

    X_shap_df = pd.DataFrame(
        X_processed,
        columns=feature_names
    )

    calibrated_model = fitted_pipe.named_steps["model"]
    calibrated_classifiers = calibrated_model.calibrated_classifiers_

    base_estimators = [
        extract_base_estimator(cc)
        for cc in calibrated_classifiers
    ]

    print(
        f"{context_name} | SHAP input shape: {X_shap_df.shape} | "
        f"Base estimators: {len(base_estimators)}"
    )

    all_shap_values = []

    for idx, base_estimator in enumerate(base_estimators, start=1):
        print(
            f"{context_name} | Computing SHAP for calibrated estimator "
            f"{idx}/{len(base_estimators)}..."
        )

        explainer = shap.TreeExplainer(base_estimator)
        raw_shap_values = explainer.shap_values(X_shap_df)

        shap_vals_class1_i = get_class1_shap_values(raw_shap_values)

        if shap_vals_class1_i.shape[1] != X_shap_df.shape[1]:
            raise ValueError(
                f"SHAP feature mismatch in {context_name}: "
                f"SHAP has {shap_vals_class1_i.shape[1]} features, "
                f"but X has {X_shap_df.shape[1]} features."
            )

        all_shap_values.append(shap_vals_class1_i)

        if COOL_DOWN_SECONDS > 0:
            time.sleep(COOL_DOWN_SECONDS)

    shap_vals_class1 = np.mean(
        np.stack(all_shap_values, axis=0),
        axis=0
    )

    output_mask = np.array([
        not is_identifier_feature_name(f)
        for f in feature_names
    ])

    feature_names = [
        f for f, keep in zip(feature_names, output_mask)
        if keep
    ]

    X_shap_df = X_shap_df.loc[:, feature_names].copy()
    shap_vals_class1 = shap_vals_class1[:, output_mask]

    return shap_vals_class1, X_shap_df, feature_names


# -----------------------------------------------------------------------------------------
# Extra) Run this SHAP block as if identifier-like columns do not exist
# -----------------------------------------------------------------------------------------
X_train_shap = remove_identifier_columns_from_input(X_train)

final_pipe = clone(final_pipe)
final_pipe = remove_identifier_features_from_pipeline(final_pipe)


# -----------------------------------------------------------------------------------------
# 3) Make sure final pipeline is fitted
# -----------------------------------------------------------------------------------------
try:
    _ = final_pipe.named_steps["model"].calibrated_classifiers_
except Exception:
    print("Pipeline was not fitted. Fitting final pipeline now...")
    final_pipe.fit(X_train_shap, y_train)


# -----------------------------------------------------------------------------------------
# 4) Final fitted model SHAP analysis on processed training data
# -----------------------------------------------------------------------------------------
shap_vals_class1, X_shap_df, feature_names = compute_shap_for_fitted_pipeline(
    final_pipe,
    X_train_shap,
    context_name="Final fitted pipeline"
)

print(f"Final SHAP input shape: {X_shap_df.shape}")
print(f"Number of final SHAP features: {len(feature_names)}")



mean_abs_shap = np.mean(
    np.abs(shap_vals_class1),
    axis=0
)

std_abs_shap = np.std(
    np.abs(shap_vals_class1),
    axis=0
)

stability_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean_SHAP": mean_abs_shap,
    "Std_SHAP_AcrossSamples": std_abs_shap
}).sort_values(
    by="Mean_SHAP",
    ascending=False
).reset_index(drop=True)

stability_df["FinalModel_Rank"] = np.arange(1, len(stability_df) + 1)

print("\n--- Final Model SHAP Importance and Across-Sample Variation ---")
print(stability_df)



print("\n>>> Starting fold-level SHAP ranking stability analysis...")

y_train_array = np.asarray(y_train)

cv = StratifiedKFold(
    n_splits=SHAP_STABILITY_N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE_SHAP
)

fold_importance_tables = []

for fold_id, (train_idx, valid_idx) in enumerate(cv.split(X_train_shap, y_train_array), start=1):
    print(f"\n--- SHAP stability fold {fold_id}/{SHAP_STABILITY_N_SPLITS} ---")

    X_fold_train = safe_index_rows(X_train_shap, train_idx)
    y_fold_train = safe_index_vector(y_train, train_idx)

    X_fold_valid = safe_index_rows(X_train_shap, valid_idx)

    if SHAP_MAX_SAMPLES_PER_FOLD is not None and len(X_fold_valid) > SHAP_MAX_SAMPLES_PER_FOLD:
        rng = np.random.RandomState(RANDOM_STATE_SHAP + fold_id)
        sampled_positions = rng.choice(
            np.arange(len(X_fold_valid)),
            size=SHAP_MAX_SAMPLES_PER_FOLD,
            replace=False
        )
        X_fold_valid = safe_index_rows(X_fold_valid, sampled_positions)

    fold_pipe = clone(final_pipe)
    fold_pipe.fit(X_fold_train, y_fold_train)

    fold_shap_vals, fold_X_shap_df, fold_feature_names = compute_shap_for_fitted_pipeline(
        fold_pipe,
        X_fold_valid,
        context_name=f"Fold {fold_id}"
    )

    fold_mean_abs_shap = np.mean(
        np.abs(fold_shap_vals),
        axis=0
    )

    fold_df = pd.DataFrame({
        "Fold": fold_id,
        "Feature": fold_feature_names,
        "MeanAbsSHAP_Fold": fold_mean_abs_shap
    })

    fold_df["Rank_Fold"] = fold_df["MeanAbsSHAP_Fold"].rank(
        ascending=False,
        method="min"
    ).astype(int)

    fold_importance_tables.append(fold_df)

    if COOL_DOWN_SECONDS > 0:
        time.sleep(COOL_DOWN_SECONDS)

fold_importance_df = pd.concat(
    fold_importance_tables,
    ignore_index=True
)

fold_importance_df["Selected_TopN"] = (
    fold_importance_df["Rank_Fold"] <= SHAP_STABILITY_TOP_N
).astype(int)

importance_matrix = fold_importance_df.pivot_table(
    index="Feature",
    columns="Fold",
    values="MeanAbsSHAP_Fold",
    aggfunc="mean"
)

rank_matrix = fold_importance_df.pivot_table(
    index="Feature",
    columns="Fold",
    values="Rank_Fold",
    aggfunc="mean"
)

selection_frequency = fold_importance_df.groupby("Feature")["Selected_TopN"].mean()

fold_stability_df = pd.DataFrame({
    "Feature": importance_matrix.index,
    "Mean_SHAP_AcrossFolds": importance_matrix.mean(axis=1, skipna=True).values,
    "Std_SHAP_AcrossFolds": importance_matrix.std(axis=1, skipna=True).values,
    "Mean_Rank_AcrossFolds": rank_matrix.mean(axis=1, skipna=True).values,
    "Std_Rank_AcrossFolds": rank_matrix.std(axis=1, skipna=True).values,
    f"Selection_Frequency_Top{SHAP_STABILITY_TOP_N}": selection_frequency.reindex(
        importance_matrix.index
    ).values,
    "Available_Folds": importance_matrix.notna().sum(axis=1).values
}).sort_values(
    by=["Mean_Rank_AcrossFolds", "Mean_SHAP_AcrossFolds"],
    ascending=[True, False]
).reset_index(drop=True)

print("\n--- Fold-Level SHAP Feature-Ranking Stability ---")
print(fold_stability_df)


# -----------------------------------------------------------------------------------------
# 7) Combine final-model SHAP table with fold-level stability table
# -----------------------------------------------------------------------------------------
combined_stability_df = stability_df.merge(
    fold_stability_df,
    on="Feature",
    how="outer"
).sort_values(
    by=["Mean_Rank_AcrossFolds", "Mean_SHAP"],
    ascending=[True, False]
).reset_index(drop=True)

print("\n--- Combined SHAP Stability Table ---")
print(combined_stability_df)


# -----------------------------------------------------------------------------------------
# 8) SHAP summary plot - final fitted model
# -----------------------------------------------------------------------------------------
plt.figure(figsize=(10, 6))

shap.summary_plot(
    shap_vals_class1,
    X_shap_df,
    feature_names=feature_names,
    show=False
)

plt.title(
    f"SHAP Feature Importance - {final_model_name} "
    "(Final Fitted Model)"
)

plt.tight_layout()
plt.savefig(
    "shap_summary_stable.png",
    dpi=300,
    bbox_inches="tight"
)
save_current_figure_all_formats("shap_summary_stable")
plt.show()


# -----------------------------------------------------------------------------------------
# 9) SHAP dependence plot for top final-model feature
# -----------------------------------------------------------------------------------------
top_feat = stability_df.iloc[0]["Feature"]

plt.figure(figsize=(8, 6))

shap.dependence_plot(
    top_feat,
    shap_vals_class1,
    X_shap_df,
    feature_names=feature_names,
    show=False
)

plt.title(
    f"SHAP Dependence Plot: {top_feat}"
)

plt.tight_layout()
plt.savefig(
    "shap_dependence_stable.png",
    dpi=300,
    bbox_inches="tight"
)
save_current_figure_all_formats("shap_dependence_stable")
plt.show()


# -----------------------------------------------------------------------------------------
# 10) Fold-level SHAP stability bar plot
# -----------------------------------------------------------------------------------------
top_fold_features = fold_stability_df.head(min(15, len(fold_stability_df))).copy()
top_fold_features = top_fold_features.sort_values(
    by="Mean_SHAP_AcrossFolds",
    ascending=True
)

plt.figure(figsize=(10, 7))

plt.barh(
    top_fold_features["Feature"],
    top_fold_features["Mean_SHAP_AcrossFolds"],
    xerr=top_fold_features["Std_SHAP_AcrossFolds"],
    capsize=3
)

plt.xlabel("Mean absolute SHAP across CV folds")
plt.ylabel("Feature")
plt.title(
    f"Fold-Level SHAP Feature-Ranking Stability - {final_model_name}"
)

plt.tight_layout()
plt.savefig(
    "shap_fold_stability_bar.png",
    dpi=300,
    bbox_inches="tight"
)
save_current_figure_all_formats("shap_fold_stability_bar")
plt.show()


# -----------------------------------------------------------------------------------------
# 11) Save SHAP stability outputs
# -----------------------------------------------------------------------------------------
stability_df.to_csv(
    "shap_stability_values.csv",
    index=False
)

fold_importance_df.to_csv(
    "shap_fold_importance_long.csv",
    index=False
)

fold_stability_df.to_csv(
    "shap_fold_stability_values.csv",
    index=False
)

rank_matrix.to_csv(
    "shap_fold_rank_matrix.csv"
)

combined_stability_df.to_csv(
    "shap_combined_stability_values.csv",
    index=False
)

print("\n SHAP Analysis completed with fold-level stability validation.")
print("• Final-model SHAP table: shap_stability_values.csv")
print("• Fold-level long table: shap_fold_importance_long.csv")
print("• Fold-level stability table: shap_fold_stability_values.csv")
print("• Fold-rank matrix: shap_fold_rank_matrix.csv")
print("• Combined stability table: shap_combined_stability_values.csv")
print("• Saved figures: shap_summary_stable.png, shap_dependence_stable.png, shap_fold_stability_bar.png")
print(f"• Additional saved figures directory: {fig_output_dir}")
print("• Conclusion: Feature-importance rankings were quantitatively assessed across stratified CV folds.")